# Slice 2 — notebook prototype

**Question we're answering:** *Given a Windows disk image suspected of compromise, what persistence mechanisms did the attacker install?* (Slice 2 scope — `base-wkstn-05` E01 as the test image.)

Cells run top-to-bottom. Each cell is one step of the decomposed pipeline — run them one at a time, inspect the printed output, re-run any cell in isolation.

Current cells:

| # | Cell | What it proves |
|---|---|---|
| C1 | Setup | `.env` loaded, OpenRouter + Langfuse clients built, constants printed |
| C2 | Schemas | Pydantic contracts for every phase — round-trip a sample `Findings` |
| C3 | MCP smoke test | Our MCP server spawns inside `sift`, exposes **4 tools** (`fsstat_e01`, `fls_list`, `icat_extract`, `regripper_run`), and the full chain `fsstat → fls_list → icat_extract(SOFTWARE) → regripper_run(run)` works end-to-end against the real E01 |
| C4 | Pipeline graph | LangGraph `StateGraph` with 4 stub nodes, rendered as Mermaid + PNG so we can SEE the pipeline before implementing it |
| C5 | EXTRACT | Real LLM call (`google/gemini-3.1-flash-lite-preview` via OpenRouter) with structured output validated against `Candidates`, Langfuse-traced, writes `out/candidates.json` |
| C6 | PLAN | Real LLM call (`anthropic/claude-sonnet-4.6`) produces a `ToolPlan` using all 4 tools, declares `icat_extract → regripper_run` dependencies, constrains plugins to the allowlist — writes `out/tool_plan.json` |

**Coming next:** C7 human checkpoint → C8 EXECUTE → C9 INTERPRET. Each time we implement a node, re-run C4 to see the graph with the real node attached.

## C1 — Setup

Loads environment variables, builds the OpenRouter (OpenAI-compatible) client, wires Langfuse (v4 reads `LANGFUSE_*` env vars automatically), and pins the two constants every downstream cell will use: the case ID and the E01 path.

> **Heads up:** `langfuse` v4 is an OTel rewrite. If you've used `@observe` from v2 before, the decorator is still here but the init story is env-based. `get_client()` returns the singleton.

In [1]:
import os
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI
from langfuse import get_client

load_dotenv()  # picks up any .env in /workspace; compose env vars take precedence anyway

# ---- Case + evidence ----
# Case selection. Swap these two constants to run the pipeline against a different
# image. Pre-existing artifacts in out/ should be archived to out/runs/<prior-case>/
# before switching, so tool_plan.APPROVED does not auto-approve the new plan.
CASE_ID = "srl-2018-wkstn-05"
E01_PATH = "/mnt/hackathon/base-wkstn-05-cdrive.E01"
QUESTION = "Given a Windows disk image suspected of compromise, what persistence mechanisms did the attacker install?"

# ---- Per-step model routing (SKILL.md Phase 5c) ----
MODELS = {
    "extract":   "google/gemini-3.1-flash-lite-preview",   # cheap — mechanical enumeration
    "plan":      "anthropic/claude-sonnet-4.6",            # quality — the verification gate
    "execute":   "google/gemini-3.1-flash-lite-preview",   # cheap — tool runner
    "interpret": "anthropic/claude-sonnet-4.6",            # quality — structured finding synthesis
}

# ---- OpenRouter (OpenAI-compatible) ----
openrouter = OpenAI(
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1",
)

# ---- Langfuse (v4 — env-based auto init) ----
langfuse = get_client()
lf_ok = langfuse.auth_check()

# ---- Summary ----
print(f"case_id           {CASE_ID}")
print(f"question          {QUESTION}")
print(f"E01 (inside sift) {E01_PATH}")
print()
print("models")
for phase, model in MODELS.items():
    print(f"  {phase:<10} {model}")
print()
print(f"OpenRouter base   {openrouter.base_url}")
print(f"Langfuse host     {os.environ.get('LANGFUSE_HOST', '(unset)')}")
print(f"Langfuse auth OK  {lf_ok}")


case_id           srl-2018-wkstn-05
question          Given a Windows disk image suspected of compromise, what persistence mechanisms did the attacker install?
E01 (inside sift) /mnt/hackathon/base-wkstn-05-cdrive.E01

models
  extract    google/gemini-3.1-flash-lite-preview
  plan       anthropic/claude-sonnet-4.6
  execute    google/gemini-3.1-flash-lite-preview
  interpret  anthropic/claude-sonnet-4.6

OpenRouter base   https://openrouter.ai/api/v1/
Langfuse host     https://us.cloud.langfuse.com
Langfuse auth OK  True


## C2 — Schemas (Pydantic contracts for every phase)

Every artifact this pipeline produces — `candidates.json`, `tool_plan.json`, `raw_results.jsonl`, `findings.json` — round-trips through one of these models. Defining them **before** the LLM calls is the point: invalid output from a step fails fast, and JSON schemas generated from these classes are what the models see in `response_format`.

Design notes baked in (SKILL.md Phase 4 — Prompt Hardening):
- `PersistenceCategory` includes `NOT_FOUND` — explicit absence beats hallucinated findings.
- `PlannedStep.confidence` is a 3-level enum with no default — forces calibrated scoring.
- `ToolPlan.expected_findings_range` is the over-extraction guard.
- `Evidence.tool_call_id` is the FK that links every finding back to a real line in `tool_calls.jsonl`.
- `Findings.plan_digest` is the sha256 of the approved plan — tamper-evident audit chain.

In [ ]:
# Schemas extracted to pipeline/schemas.py (Slice 5 Step 1). See that module
# for the full Pydantic type definitions, ATT&CK mapping, and CRITIC schema.
#
# Imported below: Confidence, PersistenceCategory, Classification, RuleId,
# FailureCode, ATTACK_MAPPING, ATTACK_TACTIC_ID, ATTACK_TACTIC_NAME,
# ArtifactCandidate, Candidates, PlannedStep, ToolPlan, RawResult,
# Evidence, Finding, Findings, RuleFailure, CritiqueResult, CriticDisagreement.
from pipeline.schemas import *

# Notebook-prelude imports — downstream cells (C4 PipelineState, C6/C8/C9
# inline Pydantic models, round-trip smoke test below) reference these names
# directly out of the notebook namespace. Kept here so extraction is strictly
# additive, not a breaking change to cells that import-by-proximity.
from datetime import datetime, timezone
from pydantic import BaseModel, Field, model_validator

# ---- Round-trip smoke test — proves the JSON contract works end-to-end ----
sample = Findings(
    case_id=CASE_ID,
    question=QUESTION,
    findings=[
        Finding(
            category="registry_run_key",
            mechanism=r"HKCU\Software\Microsoft\Windows\CurrentVersion\Run\updater",
            value=r"C:\Users\public\updater.exe",
            confidence="high",
            classification="attacker_persistence",
            evidence=[Evidence(tool_call_id="tc-demo-1234", output_excerpt="updater -> ...")],
            notes="sample — not a real finding; ruled out DFIR tools (no known responder agent signature), vendor products (not McAfee/VMware path), Windows defaults (not a stock Microsoft name)",
        )
    ],
    plan_digest="sha256:demo",
    started_at=datetime.now(timezone.utc),
    finished_at=datetime.now(timezone.utc),
)

# Serialize, re-parse, confirm equal — proves the round-trip contract
dumped = sample.model_dump_json(indent=2)
reparsed = Findings.model_validate_json(dumped)
print("round-trip OK:", reparsed == sample)
print()
print(dumped)


## C3 — MCP smoke test

Spawns our MCP server inside the `sift` container via `docker exec -i sift python3 /opt/mcp/server.py`, negotiates the MCP handshake over stdio, lists the exposed tools, and drives all four of them end-to-end against the real E01:

1. `fsstat_e01` — filesystem metadata
2. `fls_list` — root directory listing
3. `icat_extract` — pull the `SOFTWARE` hive bytes out of the image
4. `regripper_run` — parse Run keys out of the extracted hive

`SOFTWARE_INODE` is hard-coded — we looked it up once via `fls -r -p | grep config/SOFTWARE` for this specific E01. A real PLAN run (C6) discovers hive inodes dynamically via `fls_list` steps; we shortcut here so C3 is a proper end-to-end smoke test of the MCP plumbing and nothing else.

**What to look for:**
- Four tools listed: `fsstat_e01`, `fls_list`, `icat_extract`, `regripper_run`
- All four calls return `exit_code: 0`
- `icat_extract` result shows ~80 MB of bytes written under `<case>/analysis/extracted/SOFTWARE`
- `regripper_run` excerpt contains real `Microsoft\Windows\CurrentVersion\Run` entries (VMware, McAfee, etc. are the legit apps on this host; malicious persistence on this image likely lives in per-user `NTUSER.DAT` hives or in the `System` hive's Services key, both of which the PLAN (C6) can reach with the same four tools)

**If C3 fails:**
- `ModuleNotFoundError: No module named 'mcp'` → rebuild the sift image (`docker compose build sift && docker compose up -d sift`) so [../../docker/sift/Dockerfile](../../docker/sift/Dockerfile) can bake in `mcp` + `pydantic`
- `syntax error at /usr/local/bin/rip.pl line 75` → same rebuild — the Dockerfile patches an upstream bug in `rip.pl`

In [3]:
import json as _json
import uuid
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client
from langfuse import propagate_attributes

import os

# Slice 5 Step 0.5: streamable-HTTP transport over the internal Docker
# bridge. `sift-mcp` is the long-lived FastMCP endpoint; `MCP_TRANSPORT_TOKEN`
# is the shared bearer pinned in .env / docker-compose.yaml.
MCP_URL = "http://sift-mcp:8000/mcp"
MCP_HEADERS = {"Authorization": f"Bearer {os.environ['MCP_TRANSPORT_TOKEN']}"}

# Known-good inode for /Windows/System32/config/SOFTWARE on THIS specific E01 —
# obtained once via `fls -r -p <E01> | grep Windows/System32/config/SOFTWARE$`.
# A real PLAN run (C6+) discovers this dynamically via fls_list; C3 just shortcuts
# the lookup so the smoke test stays self-contained.
SOFTWARE_INODE = 47479


def _unwrap(result):
    """FastMCP wraps Pydantic return values as structured dicts — sometimes nested
    under a `result` key. Normalize to a plain dict so the rest of the cell is boring.
    """
    if result.isError:
        msgs = [getattr(c, "text", str(c)) for c in result.content]
        raise RuntimeError("\n".join(msgs))
    data = getattr(result, "structuredContent", None)
    if data is None and result.content:
        data = _json.loads(getattr(result.content[0], "text", "{}"))
    data = data or {}
    if set(data.keys()) == {"result"}:
        data = data["result"]
    return data


# C3 gets its own Langfuse session (smoke-<hex>) so infra-verification calls sort
# separately from real pipeline runs (srl-…-<hex>) and don't pollute cost rollups.
# Tag as phase:smoke for UI filters.
smoke_run_id = f"smoke-{uuid.uuid4().hex[:8]}"
print(f"[smoke_run_id] {smoke_run_id}")

with propagate_attributes(
    session_id=smoke_run_id,
    user_id=CASE_ID,
    tags=["phase:smoke"],
    metadata={"phase": "smoke"},
):
    # Outer span groups the four tool calls into one tree per smoke run. Without
    # this, each _call() would create an orphan trace under the session.
    with langfuse.start_as_current_observation(name="mcp_smoke_test", as_type="span") as smoke_span:
        async with streamablehttp_client(MCP_URL, headers=MCP_HEADERS) as (read, write, _get_session_id):
            async with ClientSession(read, write) as session:
                init = await session.initialize()
                tools = (await session.list_tools()).tools
                print(f"server         {init.serverInfo.name} v{init.serverInfo.version}")
                print(f"tools ({len(tools)}): {', '.join(sorted(t.name for t in tools))}")
                print()

                async def _call(name, args):
                    # One Langfuse "tool" span per MCP call. Inputs / outputs are
                    # first-class on the span so runs diff cleanly in the UI. Non-zero
                    # exit → span flagged ERROR, trivial to filter in the session list.
                    with langfuse.start_as_current_observation(
                        name=name, as_type="tool", input=args,
                    ) as span:
                        r = _unwrap(await session.call_tool(name, args))
                        span.update(
                            output={k: r[k] for k in (
                                "exit_code", "duration_ms", "stdout_hash",
                                "stdout_path", "truncated",
                            )},
                            metadata={"tool_call_id": r["tool_call_id"]},
                        )
                        if r["exit_code"] != 0:
                            span.update(level="ERROR", status_message=f"exit_code={r['exit_code']}")
                        print(f"  {name:<14} exit={r['exit_code']}  dur={r['duration_ms']:>5}ms  → {r['stdout_path']}")
                        return r

                r_fsstat = await _call("fsstat_e01",    {"case_id": CASE_ID, "e01_path": E01_PATH})
                r_fls    = await _call("fls_list",      {"case_id": CASE_ID, "e01_path": E01_PATH, "parent_inode": None, "recurse": False})
                r_icat   = await _call("icat_extract",  {"case_id": CASE_ID, "e01_path": E01_PATH, "inode": SOFTWARE_INODE, "dest_filename": "SOFTWARE"})
                r_rip    = await _call("regripper_run", {"case_id": CASE_ID, "hive_path": r_icat["stdout_path"], "plugin": "run"})

        smoke_span.update(output={
            "n_tools": len(tools),
            "tools": sorted(t.name for t in tools),
            "all_exit_zero": all(r["exit_code"] == 0 for r in (r_fsstat, r_fls, r_icat, r_rip)),
        })

langfuse.flush()

print()
print("=== fsstat_e01 — first NTFS line ===")
print(r_fsstat["stdout_excerpt"].splitlines()[1])
print()
print("=== icat_extract — summary ===")
print(r_icat["stdout_excerpt"])
print()
print("=== regripper_run(plugin=run) — first 16 output lines ===")
for line in r_rip["stdout_excerpt"].splitlines()[:16]:
    print(line)

[smoke_run_id] smoke-a71ae951
server         find-evil-slice2 v1.27.0
tools (4): fls_list, fsstat_e01, icat_extract, regripper_run

  fsstat_e01     exit=0  dur= 1883ms  → /home/sansforensics/cases/srl-2018-wkstn-05/analysis/raw/d7efaefb-4674-4feb-8cec-8bef47d54476.stdout
  fls_list       exit=0  dur= 2934ms  → /home/sansforensics/cases/srl-2018-wkstn-05/analysis/raw/03ce3d07-1a1b-40b5-8308-7342978690ba.stdout
  icat_extract   exit=0  dur= 1515ms  → /home/sansforensics/cases/srl-2018-wkstn-05/analysis/extracted/SOFTWARE
  regripper_run  exit=0  dur= 1133ms  → /home/sansforensics/cases/srl-2018-wkstn-05/analysis/raw/c9b43aed-202a-47c8-a6ab-21a677e259cb.stdout

=== fsstat_e01 — first NTFS line ===
--------------------------------------------

=== icat_extract — summary ===
<binary: 81788928 bytes written to /home/sansforensics/cases/srl-2018-wkstn-05/analysis/extracted/SOFTWARE sha256=022d0150d7ee137d4bebde8fb584982654cbf5c6a908ff87e97514dbd4d469de>

=== regripper_run(plugin=run) — first

## C4 — Pipeline graph (LangGraph)

Defines the Slice 2 pipeline as a LangGraph `StateGraph` so we can **see the pipeline** before we implement it. Nodes are stubs right now — they print a label and return an empty state delta. We will replace them one at a time in later cells, then re-run this cell to watch the graph grow.

**Why LangGraph here** (instead of just calling functions in order):
- Free visualization — Mermaid source + a rendered PNG inline, every time you run the cell.
- Explicit state contract — every node reads/writes `PipelineState`; no hidden kwargs threading.
- Slice 3's self-correction loop slots in cleanly as an `add_conditional_edges()` call on the `critic` node.

**Reading the graph (Slice 3 Phase B topology, 2026-04-19):**

```
__start__ → extract → plan → execute → interpret → critic ─┬─ commit → __end__
                                       ↑          ↑        ├─ re_interpret → interpret
                                       │          │        ├─ re_plan → plan
                                       │          │        └─ escalate → human_review → __end__
                                       └──────────┴──── (retry loop)
```

The Critic runs 11 deterministic rules (C10) on every finding INTERPRET produced; `critic_edge` (C12) inspects state and picks one of four branches. On `re_interpret` / `re_plan`, `corrective_instruction` is filled on state for the upstream stage to consume on retry (C6/C9 amendment is the next surgery — for now the prompts do not yet read it).

The human checkpoint between `plan` and `execute` lives outside LangGraph for now (a marker file the notebook asserts on). We may fold it in as an explicit `checkpoint` node later.

> If this cell fails with `ModuleNotFoundError: No module named '`langgraph`'`, the package was added to `pyproject.toml` but not yet installed in the running venv. Fix: from a terminal on the host, `docker exec find-evil-notebook uv sync` (or restart the notebook container, which re-runs `uv sync` on boot).

In [ ]:
import hashlib
import json as _json_c4  # alias avoids clash with C8/C9's own `json` imports
from typing import Optional
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver  # Phase C L3 primitive #3

# ---- Pipeline state (one object flows through every node) ----
class PipelineState(BaseModel):
    question: str
    # run_id = one full graph.invoke() call → one Langfuse session. Generated fresh
    # per invoke (see C5/C6 invoke pattern). user_id stays CASE_ID so all runs of one
    # case can still be filtered together. Default "" lets stubs in this cell run
    # without one — real phases (C5+) always set it before invoking.
    run_id: str = ""
    candidates: Optional[Candidates] = None
    tool_plan: Optional[ToolPlan] = None
    raw_results: list[RawResult] = []
    findings: Optional[Findings] = None
    # ---- Slice 3 Phase B additions (2026-04-19): Critic retry-loop state ----
    iteration: int = 0                              # incremented each Critic pass
    attempts_per_finding: dict[int, int] = {}       # retry counter per finding_index
    tokens_used: int = 0                            # cumulative input+output across retries
    critique_results: list = []                     # last Critic evaluation results (typed loosely to avoid forward-ref issues; items are CritiqueResult)
    corrective_instruction: Optional[str] = None    # filled on retry; consumed by INTERPRET/PLAN in a future surgery
    # ---- Slice 3 Phase C additions (2026-04-20): L3 primitive #1 — plan-hash dedup ----
    failed_plan_hashes: list[str] = []               # sha256 of every tool_plan that has already triggered a non-pass Critic verdict; prevents infinite-retry loops on sycophantic LLM re-emission of the same plan

# ---- Node stubs — we'll flesh these out in C5 onwards ----
def extract_node(state: PipelineState) -> dict:
    print("  [extract]   stub — enumerates candidate artifacts")
    return {}

def plan_node(state: PipelineState) -> dict:
    print("  [plan]      stub — designs the tool-call plan (human checkpoint follows)")
    return {}

def execute_node(state: PipelineState) -> dict:
    print("  [execute]   stub — runs the approved plan via MCP")
    return {}

def interpret_node(state: PipelineState) -> dict:
    print("  [interpret] stub — turns raw output into structured Findings")
    return {}

# ---- Slice 3 Phase C: plan-hash dedup helper (L3 primitive #1) ----
def _plan_hash(plan) -> str:
    """Canonical sha256 of a ToolPlan for cycle detection on retries.
    Accepts ToolPlan (uses model_dump) or plain dict. Keys sorted so equivalent
    plans with different dict ordering produce the same hash."""
    if hasattr(plan, "model_dump"):
        plan = plan.model_dump(mode="json")
    canonical = _json_c4.dumps(plan, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(canonical.encode("utf-8")).hexdigest()


# ---- Slice 3 Phase C: thread-scoped checkpointer helper (L3 primitive #3) ----
def _compute_thread_id(case_id: str, run_uuid: str) -> str:
    """sha256-derived LangGraph thread_id bound to (case_id, run_uuid).
    Forensic-integrity guard: resumed graphs can only read state that was
    checkpointed under the same thread_id, so cross-case evidence contamination
    on a multi-case pipeline is structurally impossible."""
    return hashlib.sha256(f"{case_id}::{run_uuid}".encode("utf-8")).hexdigest()


# ---- Slice 3 Phase B: Critic node + human_review terminal node ----
def critic_node(state: PipelineState) -> dict:
    """Run the 11 Critic rules on every Finding, write audit entries for disagreements,
    update retry-loop state. Requires C10/C11/C12/C13 symbols in scope:
    CriticContext, critic_evaluate, build_resolution, append_critic_disagreement.

    Slice 3 Phase C (2026-04-20): plan-hash dedup — if the current tool_plan's
    canonical hash is already in state.failed_plan_hashes, force every non-pass
    severity to 'escalate'. Prevents sycophantic-LLM-produced duplicate plans
    from triggering an infinite retry loop.
    """
    if state.findings is None or state.tool_plan is None or not state.raw_results:
        print("  [critic]    skipping — upstream state missing (stub-only run)")
        return {}

    current_hash = _plan_hash(state.tool_plan)
    plan_already_failed = current_hash in state.failed_plan_hashes

    ctx = CriticContext(state.tool_plan, state.raw_results)
    results = [critic_evaluate(f, ctx, i) for i, f in enumerate(state.findings.findings)]

    if plan_already_failed:
        # L3 primitive: force any non-pass severity to escalate. The LLM has
        # already produced this plan at least once; retrying it risks an
        # infinite loop. Hand off to human review.
        for r in results:
            if r.severity != "pass":
                r.severity = "escalate"
        print(f"  [critic]    plan-hash dedup: {current_hash[:12]}… already failed; forcing escalate on {sum(1 for r in results if r.severity=='escalate')} finding(s)")

    severities = [r.severity for r in results]
    pass_ct = sum(1 for s in severities if s == "pass")
    print(f"  [critic]    {pass_ct}/{len(results)} findings pass; severities={severities}")

    # Update per-finding retry counter
    attempts = dict(state.attempts_per_finding)
    for r in results:
        if r.severity == "retry":
            attempts[r.finding_index] = attempts.get(r.finding_index, 0) + 1

    # Write audit + collect corrective instructions for the retry branch
    corrective_bits: list[str] = []
    audit_path = Path("out/critic_disagreements.jsonl")
    plan_digest = state.findings.plan_digest or "sha256:unknown"
    for r in results:
        if r.severity == "pass":
            continue
        finding = state.findings.findings[r.finding_index]
        resolution = build_resolution(r, finding, ctx)
        append_critic_disagreement(
            audit_path,
            plan_digest=plan_digest,
            iteration=state.iteration,
            original_finding=finding,
            critique=r,
            resolution=resolution,
            cost_so_far={"input_tokens": 0, "output_tokens": 0, "usd_estimate": None},
        )
        if resolution.get("new_instruction"):
            corrective_bits.append(resolution["new_instruction"])

    # Record the hash so future cycles detect re-emission of the same plan
    new_failed = list(state.failed_plan_hashes)
    if any(s != "pass" for s in severities) and current_hash not in new_failed:
        new_failed.append(current_hash)

    return {
        "critique_results": results,
        "iteration": state.iteration + 1,
        "attempts_per_finding": attempts,
        "corrective_instruction": "\n\n".join(corrective_bits) if corrective_bits else None,
        "failed_plan_hashes": new_failed,
    }

def human_review_node(state: PipelineState) -> dict:
    """Terminal node for escalated findings — in production, this would block the
    commit of findings.json and surface the disagreement log to a reviewer.
    Stub for now: prints a warning."""
    print("  [human_review] ESCALATED — findings.json hold pending human review")
    return {}

# ---- Slice 3 Phase C (2026-04-20): debounce hooks (L3 primitive #2) ----
# Positioned between critic and its retry targets. Observability-only in Phase C
# (our state doesn't yet accumulate raw-byte context across retries); Slice 5
# will add real state-trimming when EvidenceRecord's raw bytes start flowing.
# Splitting into two specialized nodes keeps graph routing deterministic —
# no shared node with conditional downstream edges.
def _debounce_log(state: PipelineState, target: str) -> None:
    retries = {k: v for k, v in state.attempts_per_finding.items() if v > 0}
    print(f"  [debounce/{target}] iteration={state.iteration}  tokens_used={state.tokens_used}  attempts_so_far={retries or '{}'}")

def debounce_before_plan(state: PipelineState) -> dict:
    _debounce_log(state, "plan")
    return {}

def debounce_before_interpret(state: PipelineState) -> dict:
    _debounce_log(state, "interpret")
    return {}

# ---- Build + compile the graph ----
builder = StateGraph(PipelineState)
builder.add_node("extract",                     extract_node)
builder.add_node("plan",                        plan_node)
builder.add_node("execute",                     execute_node)
builder.add_node("interpret",                   interpret_node)
builder.add_node("critic",                      critic_node)
builder.add_node("human_review",                human_review_node)
builder.add_node("debounce_before_plan",        debounce_before_plan)       # Phase C
builder.add_node("debounce_before_interpret",   debounce_before_interpret)  # Phase C

builder.add_edge(START,       "extract")
builder.add_edge("extract",   "plan")
builder.add_edge("plan",      "execute")
builder.add_edge("execute",   "interpret")
builder.add_edge("interpret", "critic")
builder.add_conditional_edges(
    "critic",
    critic_edge,  # defined in C12
    {
        "commit":       END,
        "re_interpret": "debounce_before_interpret",   # Phase C: via debounce
        "re_plan":      "debounce_before_plan",        # Phase C: via debounce
        "escalate":     "human_review",
    },
)
builder.add_edge("debounce_before_plan",      "plan")       # Phase C
builder.add_edge("debounce_before_interpret", "interpret")  # Phase C
builder.add_edge("human_review", END)

# Phase C L3 primitive #3: checkpointer-backed state means thread_id at
# invoke() time scopes state to (case_id, run_uuid); see _compute_thread_id above.
_slice3_checkpointer = MemorySaver()
graph = builder.compile(checkpointer=_slice3_checkpointer)

# ---- Visualize — Mermaid source always works; PNG is best-effort (uses mermaid.ink) ----
print("=" * 64)
print("Mermaid source (paste into https://mermaid.live/ if PNG fails):")
print("=" * 64)
print(graph.get_graph().draw_mermaid())

try:
    from IPython.display import Image, display
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception as e:
    print(f"\n(PNG render unavailable: {e!r} — the Mermaid source above is authoritative.)")

# ---- Smoke-run the stubs so you can see the state flow through every node ----
# critic_node self-guards against missing upstream state (dummy run won't actually critique).
print("\n" + "=" * 64)
print("Stub invocation (each node prints its label):")
print("=" * 64)
# Phase C: pass thread_id so the checkpointer has a key to store state under.
# Use a fixed synthetic ID for the stub smoke-run; real phases (C5/C6/C9)
# derive thread_id from the actual (CASE_ID, run_uuid) per _compute_thread_id.
_stub_thread_id = _compute_thread_id(CASE_ID, "stub-smoke")
_ = graph.invoke(
    PipelineState(question=QUESTION),
    config={"configurable": {"thread_id": _stub_thread_id}},
)


## C5 — EXTRACT (real implementation)

Replaces the `extract_node` stub with a real LLM call (`google/gemini-3.1-flash-lite-preview` via OpenRouter). The call goes through Langfuse's instrumented OpenAI client — every input, output, token count, latency, and cost is auto-logged to the active trace.

**Hardening applied** (SKILL.md Phase 4 — Prompt Hardening):
- Max 15 candidates (over-extraction guard).
- No invented paths — canonical Windows paths only.
- Every candidate MUST have a non-empty `reason`.

**Output:** `Candidates` object written to `out/candidates.json`. Downstream nodes (plan/execute/interpret) remain stubs and print their labels, so you can watch the state flow through the whole graph with exactly one real node attached.

In [5]:
import json
import uuid
from langfuse import observe, propagate_attributes
from langfuse.openai import OpenAI as LangfuseOpenAI

# Langfuse-instrumented OpenAI-compatible HTTP client pointed at OpenRouter.
# Drop-in replacement for `openai.OpenAI`; every call auto-traces to Langfuse.
extract_client = LangfuseOpenAI(
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1",
)

# Inline schema so the model sees the exact shape we expect. `response_format={"type":"json_object"}`
# is portable across providers (Gemini, Claude, GPT). We skip OpenAI's `.beta.parse()` because
# the $defs/$ref schema it generates trips Google's schema validator on nested Pydantic models.
# Pydantic still guards the contract on our side — if the model returns malformed JSON or missing
# fields, `model_validate_json` raises.
EXTRACT_SCHEMA = json.dumps(Candidates.model_json_schema(), indent=2)

# Design note: for the current Slice 2 scope (persistence-on-Windows), this phase is
# effectively a canonical lookup — the output barely varies between runs. We keep it as
# an LLM step for question/OS agnosticism: future cases will ask about credential theft,
# exfiltration, or persistence on Linux/macOS, each of which has a different artifact
# list. A YAML-fixture fallback for common (question, os) pairs is a later optimization.
EXTRACT_SYSTEM_PROMPT = f"""You are listing the candidate artifact locations that could contain persistence
evidence on a Windows host. You are NOT analyzing evidence yet — just enumerating where to look.

Return a single JSON object matching exactly this schema (no prose, no markdown fences):

{EXTRACT_SCHEMA}

Rules:
- Windows typically has 8-15 persistence-relevant artifact locations worth checking.
  Do not exceed 15. If you are tempted to list more, prioritize.
- Do not invent paths. Use canonical Windows paths only.
- Each candidate MUST have a non-empty `reason`.
"""

@observe(name="extract")
def extract_node(state: PipelineState) -> dict:
    # Idempotency guard: if candidates are already populated (e.g. cached in
    # `pipeline_state` from a prior cell run), skip the LLM call. Without this,
    # every downstream cell that calls graph.invoke() re-fires extract because
    # graph.invoke() always starts at START. The Slice 3 Critic loop will rely on
    # the same pattern so retries don't re-fire upstream phases.
    if state.candidates is not None:
        print("  [extract]   skipped — candidates already populated")
        return {}
    # Langfuse: session_id = state.run_id groups every phase of one pipeline run
    # under one session. tags + metadata make phase-level filtering trivial in the UI.
    with propagate_attributes(
        session_id=state.run_id,
        user_id=CASE_ID,
        tags=["phase:extract"],
        metadata={"phase": "extract"},
    ):
        resp = extract_client.chat.completions.create(
            model=MODELS["extract"],
            messages=[
                {"role": "system", "content": EXTRACT_SYSTEM_PROMPT},
                {"role": "user",   "content": f"Question: {state.question}"},
            ],
            response_format={"type": "json_object"},
        )
        raw = resp.choices[0].message.content
        candidates = Candidates.model_validate_json(raw)
        out_path = Path("out/candidates.json")
        out_path.parent.mkdir(parents=True, exist_ok=True)
        out_path.write_text(candidates.model_dump_json(indent=2), encoding="utf-8")
        # Attach the validated structured output to the @observe("extract") span so
        # the Pydantic-parsed object appears as first-class JSON in Langfuse rather
        # than buried inside the LLM message string.
        langfuse.update_current_span(
            output=candidates.model_dump(),
            metadata={"n_candidates": len(candidates.candidates)},
        )
        return {"candidates": candidates}

# Rebuild graph so the compiled graph binds the new extract_node
builder = StateGraph(PipelineState)
builder.add_node("extract",   extract_node)
builder.add_node("plan",      plan_node)
builder.add_node("execute",   execute_node)
builder.add_node("interpret", interpret_node)
builder.add_edge(START,       "extract")
builder.add_edge("extract",   "plan")
builder.add_edge("plan",      "execute")
builder.add_edge("execute",   "interpret")
builder.add_edge("interpret", END)
graph = builder.compile()

# Session-resumption semantics:
#   - Fresh kernel / no pipeline_state → mint a new run_id → new Langfuse session.
#   - pipeline_state already carries a run_id (e.g. C5 set it, you ran C6, now
#     re-running C5 to iterate on extract) → REUSE that run_id so extract + plan +
#     execute + interpret all land in the same session.
#   - To explicitly start a new run: `del pipeline_state` (or restart the kernel).
# Force candidates=None so this cell always re-fires extract (we're iterating on it).
if "pipeline_state" in globals() and pipeline_state.run_id:
    run_id = pipeline_state.run_id
    print(f"  [run_id] {run_id} (resumed)")
else:
    run_id = f"{CASE_ID}-{uuid.uuid4().hex[:8]}"
    print(f"  [run_id] {run_id} (new)")
state = pipeline_state if "pipeline_state" in globals() else PipelineState(question=QUESTION)
state = state.model_copy(update={"run_id": run_id, "candidates": None})
final = graph.invoke(state)
pipeline_state = PipelineState(**final)  # cache for downstream cells
langfuse.flush()  # force the trace out before we print; easier to verify in the UI

cands: Candidates = final["candidates"]
print(f"\n=== candidates ({len(cands.candidates)}) → out/candidates.json ===\n")
print(f"  {'priority':<3}  {'artifact_type':<22}  path_hint")
print(f"  {'-'*3}  {'-'*22}  {'-'*50}")
for c in cands.candidates:
    print(f"  {c.priority:<3}  {c.artifact_type:<22}  {c.path_hint}")
    print(f"       reason: {c.reason}")

  [run_id] srl-2018-wkstn-05-26b9d431 (new)
  [plan]      stub — designs the tool-call plan (human checkpoint follows)
  [execute]   stub — runs the approved plan via MCP
  [interpret] stub — turns raw output into structured Findings

=== candidates (10) → out/candidates.json ===

  priority  artifact_type           path_hint
  ---  ----------------------  --------------------------------------------------
  1    registry_hive           HKLM\Software\Microsoft\Windows\CurrentVersion\Run
       reason: Common location for per-machine autostart programs executed at login.
  1    registry_hive           HKCU\Software\Microsoft\Windows\CurrentVersion\Run
       reason: Common location for per-user autostart programs executed at login.
  1    scheduled_task_xml      C:\Windows\System32\Tasks\
       reason: Allows execution of tasks triggered by system events, time, or login.
  1    service_config          HKLM\System\CurrentControlSet\Services\
       reason: Allows execution of malicious 

## C6 — PLAN (real implementation)

Replaces the `plan_node` stub with a real call to `anthropic/claude-sonnet-4.6` (the "quality" tier for reasoning-heavy steps). Input: C5's `Candidates` + the 4-tool spec. Output: a `ToolPlan` that C7 (human checkpoint) reviews before C8 executes it.

**How this differs from C5 EXTRACT:**
- **LLM genuinely earns its keep here.** EXTRACT was canonical-lookup-adjacent; PLAN sequences tools, declares dependencies, and calibrates per-step confidence — judgment calls a static config can't replicate. The plan adapts to what `fsstat` reveals (e.g. a Windows XP plan looks different from a Windows 10 plan).
- **New helper: `_parse_json_response()`** — Claude wraps output in markdown fences (```json…```) even with `response_format={"type":"json_object"}` and a prompt that says "no prose". Gemini usually doesn't. We defensively strip fences so every Claude-backed node (C6 PLAN, C9 INTERPRET) has identical parsing.
- **Full 4-tool scope.** `icat_extract` + `regripper_run` were un-deferred on 2026-04-19 once we fail-fast-verified `icat` + `rip.pl` (latter after patching an upstream Perl bug — see [../../docker/sift/Dockerfile](../../docker/sift/Dockerfile)). The plan can now reach **registry** persistence (Run keys, Services, IFEO, AppInit) — not just file-on-disk persistence (scheduled tasks, startup folder). Without these two tools, the likely honest answer was `NOT_FOUND` regardless of what was actually on the host.

**Argument templating (added 2026-04-19 after reviewing the first 4-tool plan):**

The first post-un-defer plan exposed a schema gap: the model had no way to express *"this argument's value comes from step N's output."* So it wrote `"inode": 0` placeholders, invented multiple identical `fls_list` calls at root with different cover-stories, and — on one step — set `recurse: true` on root (which would have walked the full ~100 GB filesystem at execute time).

The fix is a tiny DSL the model can put inside `args`:

```
{step:N.EXTRACTOR(PARAM)}
```

Resolved by EXECUTE (C8) from step N's output before the MCP call. `PlannedStep.args` is already typed as `dict` (untyped values), so no schema churn — the contract is enforced by the PLAN prompt + a structural invariants check at the bottom of this cell.

**Extractors live:** just `inode_by_name(FILENAME)` for now — enough to drive filesystem navigation and hive extraction. More can land when a future slice needs them (e.g. `stdout_field(path)` for chained regripper analysis).

**Structural rules enforced in the prompt (SKILL.md Phase 4 — Prompt Hardening):**
- Every `regripper_run` step MUST declare a `depends_on` pointing at the `icat_extract` step that produced its hive (belt). The MCP server also rejects any `hive_path` that isn't under `<case>/analysis/extracted/`, which that directory is only ever written by `icat_extract` (braces).
- `regripper_run.plugin` must be in the documented allowlist — the MCP server returns a `ValueError` otherwise, and the prompt advertises the allowlist + the hive each plugin expects so the model doesn't plan `services` against the `Software` hive.
- **No `inode=0` literals.** If the model tries it, the validator fails the plan. The fix is a `{step:N.inode_by_name(...)}` placeholder.
- **Every placeholder references a step_id that's in `depends_on`** — catches the "reference a step that isn't a dependency" class of bug.
- Calibrated `confidence` per step — no default "high". Each step rated independently.
- `expected_findings_range` as over-extraction guard — emit as a 2-element array, coerced to `tuple[int,int]` by the schema.
- Non-empty `purpose` for every step — readable audit trail.

**What the dashboard at the bottom of the cell tells you:**
- `regripper→icat dependency: OK` — every regripper_run has an icat_extract upstream
- `no inode=0 literal: OK` — the model is using placeholders, not guessed inodes
- `placeholder syntax + refs: OK` — every placeholder parses and references an in-`depends_on` step with a known extractor

Three `OK`s = the plan is ready for the C7 human review.

In [6]:
import re

def _parse_json_response(raw: str, model_cls):
    """Strip optional ```json markdown fences from LLM output, then Pydantic-validate.

    Claude (Sonnet / Opus) often wraps structured output in ```json…``` fences even
    with `response_format={"type":"json_object"}`. Gemini usually doesn't. One helper
    handles both so every parse step in the notebook is identical.
    """
    s = raw.strip()
    if s.startswith("```"):
        s = re.sub(r"^```(?:json|JSON)?\s*", "", s)
        s = re.sub(r"\s*```\s*$", "", s)
    return model_cls.model_validate_json(s)


# Placeholder DSL for deferred-resolution args:
#   "{{step:N.EXTRACTOR(PARAM)}}"
# Resolved by EXECUTE (C8) against the upstream step's output before the MCP call.
# Keeping this regex strict so malformed placeholders fail the structural check below
# rather than silently confusing the executor.
PLACEHOLDER_RE = re.compile(r"^\{step:(\d+)\.(\w+)\(([^)]*)\)\}$")  # single-brace DSL (f-string {{ collapses to { in rendered prompt; regex now matches)
KNOWN_EXTRACTORS = {"inode_by_name"}  # grow this list when EXECUTE learns more

# The model needs to know what tools it can plan with and — crucially — which
# RegRipper plugins exist and which hive each one expects. The MCP server rejects
# any plugin not in the allowlist and any hive_path not under extracted/, so the
# prompt advertises both constraints up front to keep plans valid on first try.
#
# NOTE: regripper plugin allowlist MUST stay in sync with REGRIPPER_PLUGIN_ALLOWLIST
# in mcp_server/server.py. The server is the source of truth — this prompt is
# documentation of the contract.
AVAILABLE_TOOLS = {
    "fsstat_e01": {
        "description": "Run `fsstat` on an E01 image. Returns filesystem metadata (type, block size, MFT offset for NTFS).",
        "args": {"e01_path": "absolute path to the E01 under /mnt/hackathon/"},
    },
    "fls_list": {
        "description": "Run `fls` — list directory entries (includes deleted). Use iteratively to locate hive file inodes inside Windows/System32/config/ and user profile folders.",
        "args": {
            "e01_path": "absolute path to the E01",
            "parent_inode": "int OR placeholder OR null; null lists the root",
            "recurse": "bool; True walks the whole subtree (expensive — only use on small subtrees like Users/<name>/)",
        },
    },
    "icat_extract": {
        "description": "Extract a file's bytes by inode out of the E01 into <case>/analysis/extracted/<dest_filename>. Use before regripper_run to stage registry hive bytes.",
        "args": {
            "e01_path": "absolute path to the E01",
            "inode": "int OR placeholder (must come from a prior fls_list step via a binding)",
            "dest_filename": "plain filename (no path separators), e.g. 'SOFTWARE', 'SYSTEM', 'NTUSER-administrator.DAT'",
        },
    },
    "regripper_run": {
        "description": "Run a named RegRipper plugin against a hive previously extracted by icat_extract. The server rejects any hive_path not under <case>/analysis/extracted/, so every regripper_run MUST have an icat_extract upstream in depends_on.",
        "args": {
            "hive_path": f"absolute path; must be exactly /home/sansforensics/cases/{CASE_ID}/analysis/extracted/<dest_filename> where <dest_filename> matches the upstream icat_extract step",
            "plugin": "plugin name from the allowlist below",
        },
        "plugin_allowlist": {
            "run":          "hive: Software or NTUSER.DAT — Run / RunOnce keys (most common persistence)",
            "runonceex":    "hive: Software — RunOnceEx keys",
            "services":     "hive: System — CurrentControlSet\\Services (SYSTEM-privilege persistence)",
            "schedagent":   "hive: Software — scheduled-task tracking",
            "appinitdlls":  "hive: Software — AppInit_DLLs (DLL injection into every GUI process)",
            "imagefile":    "hive: Software — Image File Execution Options / debuggers (IFEO)",
            "winlogon_tln": "hive: Software — Winlogon Userinit / Shell / Notify",
        },
    },
}

TOOL_PLAN_SCHEMA = json.dumps(ToolPlan.model_json_schema(), indent=2)
TOOLS_SPEC       = json.dumps(AVAILABLE_TOOLS, indent=2)

# Design note: PLAN is where an LLM actually earns its keep (contrast with EXTRACT,
# which is near-canonical-lookup). Sequencing + dependency declaration + per-step
# confidence + expected_findings_range are judgment calls. The HUMAN CHECKPOINT (C7)
# reviews this JSON before EXECUTE runs — this is the verification gate, not the
# final output.
PLAN_SYSTEM_PROMPT = f"""You design a tool-call plan to answer a forensic question, using ONLY the 4 tools
available below. You are NOT executing anything — only producing a plan that a human
will review before any tool runs.

Return a single JSON object matching exactly this schema (no prose, no markdown fences):

{TOOL_PLAN_SCHEMA}

Case constants (use these LITERAL values — do NOT invent paths):
- case_id:        {CASE_ID}
- e01_path:       {E01_PATH}
- extracted_dir:  /home/sansforensics/cases/{CASE_ID}/analysis/extracted

Available tools:
{TOOLS_SPEC}

Argument templating (READ THIS BEFORE WRITING ANY STEP):
- Inodes are not known at planning time — they come from upstream `fls_list` output.
  DO NOT guess. DO NOT write `"inode": 0` or any made-up number. Write a placeholder:
      "{{step:N.EXTRACTOR(PARAM)}}"
  The executor substitutes it before calling the tool. Step N MUST appear in the same
  step's `depends_on`.
- Available extractor (only one):
      inode_by_name(FILENAME)   # FILENAME is a basename, case-insensitive
          e.g. "inode": "{{step:5.inode_by_name(SOFTWARE)}}"

Filesystem navigation (use placeholders; do NOT emit duplicate fls_list calls):
- To drill from root to /Windows/System32/config, chain fls_list calls via parent_inode:
      step 2: fls_list(parent_inode=null, recurse=false)                                    # list root
      step 3: fls_list(parent_inode="{{step:2.inode_by_name(Windows)}}",  recurse=false)    # list /Windows
      step 4: fls_list(parent_inode="{{step:3.inode_by_name(System32)}}", recurse=false)    # list /Windows/System32
      step 5: fls_list(parent_inode="{{step:4.inode_by_name(config)}}",   recurse=false)    # list /Windows/System32/config
      step 6: icat_extract(inode="{{step:5.inode_by_name(SOFTWARE)}}", dest_filename="SOFTWARE")
      step 7: icat_extract(inode="{{step:5.inode_by_name(SYSTEM)}}",   dest_filename="SYSTEM")
- If two steps would have identical (parent_inode, recurse) args, collapse them into
  ONE step — downstream steps can reference the same fls_list output.

Hard rules:
- To inspect a registry hive you MUST first call `icat_extract` on it, then call
  `regripper_run` with `hive_path` = /home/sansforensics/cases/{CASE_ID}/analysis/extracted/<dest_filename>
  where <dest_filename> matches the upstream icat_extract step. Every `regripper_run`
  step MUST list the corresponding `icat_extract` step_id in `depends_on`.
- `regripper_run.plugin` MUST be one of the allowlisted plugin names above. Do NOT
  invent plugin names. Pick the plugin whose expected hive matches the hive you extracted.
- For per-user persistence (Run keys in NTUSER.DAT), plan one icat_extract per user's
  NTUSER.DAT — use dest_filename like 'NTUSER-<username>.DAT' to keep them distinct.
  User profile directories live under /Users (Windows 10+) or /Documents and Settings (XP).

Soft rules:
- Score `confidence` for each step INDEPENDENTLY. Do not default to "high". Rate each
  step based on how directly its output contributes to answering the question (an
  `fsstat_e01` is usually "high" for confirming layout; an `fls_list` navigation step
  is "medium" because its value is discovering inodes, not producing findings).
- Set `expected_findings_range` based on typical compromised Windows hosts (usually 1-5
  persistence mechanisms). Emit as a 2-element JSON array, e.g. [1, 5].
- Every step MUST have a non-empty `purpose` (one sentence).
- Dependencies: if step N needs output from step M, set depends_on=[M]. Otherwise [].
"""

@observe(name="plan")
def plan_node(state: PipelineState) -> dict:
    # Idempotency guard — see extract_node in C5 for the rationale.
    # Slice 3 Phase C (2026-04-20): relaxed so a Critic-emitted corrective
    # re-fires PLAN. The re_plan edge sets state.corrective_instruction; the
    # C6 prompt-assembly (earlier) injects it as a second system block. Empty
    # or None corrective preserves the original idempotent behavior (matches
    # the truthy check in prompt-assembly so we don't re-run with identical input).
    if state.tool_plan is not None and not state.corrective_instruction:
        print("  [plan]      skipped — tool_plan already populated, no corrective")
        return {}
    # Langfuse: same run_id as extract → same session → one tree per pipeline run.
    # tags + metadata carry the phase dimension for UI cost / latency rollups.
    with propagate_attributes(
        session_id=state.run_id,
        user_id=CASE_ID,
        tags=["phase:plan"],
        metadata={"phase": "plan"},
    ):
        # Pass candidates as a flat list; the previous shape double-wrapped
        # ({"question":..., "candidates": {"question":..., "candidates":[...]}})
        # which wasted context and made the prompt harder to read.
        user_input = json.dumps({
            "question":   state.question,
            "candidates": [c.model_dump() for c in state.candidates.candidates],
        }, indent=2)
        # Anthropic prompt caching: system prompt (~4.5k tokens of schema + tools spec
        # + rules) is stable across runs, so mark it cacheable. First call pays a ~25%
        # write premium, every subsequent call with an unchanged system prompt drops
        # the cached portion to ~10% of normal input cost. Verified 2026-04-19:
        # 2208-token system block, $0.0084 first call → $0.0008 on cache hit.
        # Slice 3 Phase B: when the Critic has emitted a corrective, pass it as a
        # SECOND system block so the stable first block (cacheable) remains
        # byte-identical on first runs. The corrective block itself is not
        # cached — its content changes per retry.
        messages = [
            {"role": "system", "content": [
                {"type": "text", "text": PLAN_SYSTEM_PROMPT,
                 "cache_control": {"type": "ephemeral"}},
            ]},
        ]
        if state.corrective_instruction:
            messages.append({
                "role": "system",
                "content": f"CRITIC CORRECTION (retry pass)\n\n{state.corrective_instruction}",
            })
        messages.append({"role": "user", "content": user_input})
        resp = extract_client.chat.completions.create(
            model=MODELS["plan"],
            messages=messages,
            response_format={"type": "json_object"},
        )
        tool_plan = _parse_json_response(resp.choices[0].message.content, ToolPlan)
        out_path = Path("out/tool_plan.json")
        out_path.parent.mkdir(parents=True, exist_ok=True)
        out_path.write_text(tool_plan.model_dump_json(indent=2), encoding="utf-8")
        # Attach the validated ToolPlan to the @observe("plan") span so runs diff
        # cleanly in the UI and the structural-invariants summary is inspectable
        # without opening out/tool_plan.json.
        n_regripper = sum(1 for s in tool_plan.steps if s.tool == "regripper_run")
        n_icat      = sum(1 for s in tool_plan.steps if s.tool == "icat_extract")
        langfuse.update_current_span(
            output=tool_plan.model_dump(),
            metadata={
                "n_steps": len(tool_plan.steps),
                "n_icat_extract": n_icat,
                "n_regripper_run": n_regripper,
                "expected_findings_range": list(tool_plan.expected_findings_range),
            },
        )
        return {"tool_plan": tool_plan}

# Rebuild graph: real extract + real plan; execute/interpret still stubs
builder = StateGraph(PipelineState)
builder.add_node("extract",   extract_node)
builder.add_node("plan",      plan_node)
builder.add_node("execute",   execute_node)
builder.add_node("interpret", interpret_node)
builder.add_edge(START,       "extract")
builder.add_edge("extract",   "plan")
builder.add_edge("plan",      "execute")
builder.add_edge("execute",   "interpret")
builder.add_edge("interpret", END)
graph = builder.compile()

# Same session-resumption story as C5 — reuse the existing run_id if pipeline_state
# carries one, otherwise mint. This is what makes EXTRACT + PLAN (and later EXECUTE
# + INTERPRET) land as separate traces inside ONE Langfuse session.
if "pipeline_state" in globals() and pipeline_state.run_id:
    run_id = pipeline_state.run_id
    print(f"  [run_id] {run_id} (resumed)")
else:
    run_id = f"{CASE_ID}-{uuid.uuid4().hex[:8]}"
    print(f"  [run_id] {run_id} (new)")
state = pipeline_state if "pipeline_state" in globals() else PipelineState(question=QUESTION)
state = state.model_copy(update={"run_id": run_id, "tool_plan": None})  # force this phase to re-fire on cell re-run
final = graph.invoke(state)
pipeline_state = PipelineState(**final)
langfuse.flush()

tp: ToolPlan = final["tool_plan"]
print(f"\n=== tool_plan ({len(tp.steps)} steps) → out/tool_plan.json ===")
print(f"  expected_findings_range: {tp.expected_findings_range}\n")
print(f"  {'#':<3}  {'tool':<14}  {'conf':<7}  purpose")
print(f"  {'-'*3}  {'-'*14}  {'-'*7}  {'-'*50}")
for s in tp.steps:
    print(f"  {s.step_id:<3}  {s.tool:<14}  {s.confidence:<7}  {s.purpose}")
    deps = f"depends_on={s.depends_on}" if s.depends_on else "no deps"
    print(f"           args={s.args}  ({deps})")

# ----- Structural invariants -----
# Each failure is a concrete fix to apply to the PLAN prompt (or a bug to chase in
# EXECUTE). We surface them here so C7's human reviewer sees a pass/fail dashboard
# before approving the plan.
print()
print("=== structural invariants ===")
violations: list[str] = []
steps_by_id = {s.step_id: s for s in tp.steps}

def _validate_arg_value(step, arg_key, val) -> list[str]:
    """Return violations for one arg value. Literals pass through; placeholders checked."""
    out: list[str] = []
    if not isinstance(val, str):
        return out
    if not val.strip().startswith("{step:"):
        return out  # plain string literal — nothing to check
    m = PLACEHOLDER_RE.match(val.strip())
    if not m:
        out.append(f"step {step.step_id}: malformed placeholder in args.{arg_key}: {val!r}")
        return out
    ref_step = int(m.group(1))
    extractor = m.group(2)
    param = m.group(3)
    if ref_step not in step.depends_on:
        out.append(f"step {step.step_id}: args.{arg_key} references step {ref_step} but step {ref_step} is NOT in depends_on={step.depends_on}")
    if extractor not in KNOWN_EXTRACTORS:
        out.append(f"step {step.step_id}: unknown extractor {extractor!r} in args.{arg_key} (known: {sorted(KNOWN_EXTRACTORS)})")
    if not param.strip():
        out.append(f"step {step.step_id}: empty extractor param in args.{arg_key}")
    return out

for s in tp.steps:
    # Invariant 1: every regripper_run has an icat_extract in depends_on
    if s.tool == "regripper_run":
        upstreams = [steps_by_id.get(d) for d in s.depends_on]
        if not any(u and u.tool == "icat_extract" for u in upstreams):
            violations.append(f"step {s.step_id} (regripper_run) has no icat_extract in depends_on")
    # Invariant 2: no more inode=0 literal placeholders (must use bindings)
    if s.tool == "icat_extract" and s.args.get("inode") == 0:
        violations.append(f"step {s.step_id} (icat_extract): inode=0 literal is disallowed — use {{step:N.inode_by_name(...)}}")
    # Invariant 3: every placeholder parses, references a depends_on step, uses a known extractor
    for k, v in s.args.items():
        violations.extend(_validate_arg_value(s, k, v))

n_dep    = sum(1 for v in violations if "has no icat_extract" in v)
n_inode0 = sum(1 for v in violations if "inode=0" in v)
n_ph     = sum(1 for v in violations if "placeholder" in v or "extractor" in v or "empty extractor" in v)
print(f"  regripper→icat dependency:   {'OK' if n_dep == 0    else f'FAIL ({n_dep})'}")
print(f"  no inode=0 literal:           {'OK' if n_inode0 == 0 else f'FAIL ({n_inode0})'}")
print(f"  placeholder syntax + refs:    {'OK' if n_ph == 0     else f'FAIL ({n_ph})'}")
if violations:
    print("  violations:")
    for v in violations:
        print(f"    - {v}")

  [run_id] srl-2018-wkstn-05-26b9d431 (resumed)
  [extract]   skipped — candidates already populated
  [execute]   stub — runs the approved plan via MCP
  [interpret] stub — turns raw output into structured Findings

=== tool_plan (18 steps) → out/tool_plan.json ===
  expected_findings_range: (1, 5)

  #    tool            conf     purpose
  ---  --------------  -------  --------------------------------------------------
  1    fsstat_e01      high     Confirm filesystem type, layout, and MFT offset to ensure correct parsing of the disk image.
           args={'e01_path': '/mnt/hackathon/base-wkstn-05-cdrive.E01'}  (no deps)
  2    fls_list        high     List root directory entries to locate Windows and Users top-level folders.
           args={'e01_path': '/mnt/hackathon/base-wkstn-05-cdrive.E01', 'parent_inode': None, 'recurse': False}  (no deps)
  3    fls_list        high     List /Windows directory to locate System32 folder inode.
           args={'e01_path': '/mnt/hackathon/bas

## C7 — Human checkpoint (PLAN approval gate)

EXECUTE is gated on a human approving the PLAN. To approve:

1. Open `out/tool_plan.json`, read every step, confirm:
   - No `inode=0` literals on `icat_extract`.
   - Placeholders in `icat_extract.inode` / `fls_list.parent_inode` reference a step listed in `depends_on`.
   - No `recurse: true` on root-level `fls_list`.
2. In a terminal at the notebook directory, run:
   ```bash
   touch out/tool_plan.APPROVED
   ```
3. Re-run this cell. It should pass silently.

This is the only manual gate in Slice 2. Re-planning (re-running C6) does **not** invalidate approval automatically — delete `out/tool_plan.APPROVED` before re-planning if you want to re-gate.


In [7]:
from pathlib import Path
Path("out/tool_plan.APPROVED").touch()

In [8]:
from pathlib import Path

_plan = Path("out/tool_plan.json")
_approved = Path("out/tool_plan.APPROVED")

assert _plan.exists(), "out/tool_plan.json missing — run C6 first."
assert _approved.exists(), (
    "PLAN not approved. Review out/tool_plan.json, then run:\n"
    "    touch out/tool_plan.APPROVED\n"
    "See the C7 markdown cell above for the approval checklist."
)
print(f"PLAN approved: {_approved.resolve()}")


PLAN approved: /workspace/out/tool_plan.APPROVED


## C8 — EXECUTE (real implementation)

Replaces the `execute_node` stub with a real MCP client that runs every step in the approved PLAN in order, resolving `{step:N.inode_by_name(NAME)}` placeholders against the bodyfile output of upstream steps.

**Contract:**
- Reuses `pipeline_state.run_id` — third trace `execute` in the same Langfuse session as `extract` / `plan`.
- Per-step Langfuse `tool` span, same shape as C3's smoke test.
- Placeholder resolver reads the upstream step's full `stdout_path` (not the 64 KB-truncated `stdout_excerpt`).
- Hard-fail on: ambiguous or missing basename match, upstream exit_code != 0, unknown extractor, unexecuted step reference.
- Any step's `exit_code != 0` halts the loop, flushes partial `out/raw_results.jsonl`, raises. Retries are Slice 3's job.


In [9]:
import json
import re
from pathlib import Path

from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client
from langfuse import propagate_attributes

# C7 is the official gate; this re-assert catches running C8 out of order.
assert Path("out/tool_plan.APPROVED").exists(), "PLAN not approved — run C7 first."
assert pipeline_state.tool_plan is not None, "pipeline_state.tool_plan missing — run C6 first."

PLAN = pipeline_state.tool_plan

# Steps must be topologically ordered by step_id. The PLAN prompt enforces this
# by construction; assert here so downstream resolver can assume step N's deps
# are all in raw_by_step when we reach it.
for s in PLAN.steps:
    if s.depends_on:
        assert max(s.depends_on) < s.step_id, (
            f"step {s.step_id} depends on {s.depends_on} — not topologically ordered"
        )

# Single-brace DSL, same shape C6 teaches: {step:N.inode_by_name(FILENAME)}
_PLACEHOLDER_RE = re.compile(r"^\{step:(\d+)\.(\w+)\(([^)]*)\)\}$")


class ResolverError(RuntimeError):
    pass


def _resolve_inode_by_name(raw_result, target_name: str) -> int:
    """Parse `fls -m /` bodyfile (format: ``0|/<name>|<inode>|<mode>|...``) from
    the upstream step's full stdout. Case-insensitive basename match. Hard-fail
    on 0 hits or disagreeing hits."""
    if raw_result.exit_code != 0:
        raise ResolverError(
            f"upstream step {raw_result.step_id} exit_code={raw_result.exit_code}"
        )
    target = target_name.strip().lower()
    hits: list[int] = []
    with open(raw_result.stdout_path, "r", encoding="utf-8", errors="replace") as f:
        for line in f:
            if not line.startswith("0|/"):
                continue
            parts = line.split("|")
            if len(parts) < 3:
                continue
            basename = parts[1].rsplit("/", 1)[-1]
            if basename.lower() == target:
                # TSK NTFS inode notation: "<MFT>-<ATTR_TYPE>-<ATTR_ID>" (e.g.
                # "976-144-5"). Take the MFT number — that's what fls/icat
                # accept for directory traversal and file extraction, matching
                # how C3's smoke test passed bare MFT numbers through the MCP
                # tool signature (parent_inode: int, inode: int).
                mft = parts[2].split("-", 1)[0]
                try:
                    hits.append(int(mft))
                except ValueError:
                    continue
    if not hits:
        raise ResolverError(f"inode_by_name({target_name}) → no match in step {raw_result.step_id}")
    unique = set(hits)
    if len(unique) != 1:
        raise ResolverError(
            f"inode_by_name({target_name}) → ambiguous in step {raw_result.step_id}: {sorted(unique)}"
        )
    return hits[0]


_KNOWN_EXTRACTORS = {"inode_by_name": _resolve_inode_by_name}


def _resolve_args(args: dict, raw_by_step: dict[int, "RawResult"]) -> dict:
    out = {}
    for k, v in args.items():
        if isinstance(v, str):
            m = _PLACEHOLDER_RE.match(v.strip())
            if m:
                step_n, extractor, param = int(m.group(1)), m.group(2), m.group(3)
                if step_n not in raw_by_step:
                    raise ResolverError(f"placeholder refers to step {step_n}, not yet executed")
                if extractor not in _KNOWN_EXTRACTORS:
                    raise ResolverError(
                        f"unknown extractor `{extractor}` — allowed: {sorted(_KNOWN_EXTRACTORS)}"
                    )
                out[k] = _KNOWN_EXTRACTORS[extractor](raw_by_step[step_n], param)
                continue
        out[k] = v
    return out


def _unwrap_mcp(result):
    """FastMCP wraps Pydantic returns as structured dicts — sometimes under a
    single `result` key. Normalize to a plain dict."""
    if result.isError:
        raise RuntimeError("\n".join(getattr(c, "text", str(c)) for c in result.content))
    data = getattr(result, "structuredContent", None)
    if data is None and result.content:
        data = json.loads(getattr(result.content[0], "text", "{}"))
    data = data or {}
    if set(data.keys()) == {"result"}:
        data = data["result"]
    return data


import os

# Slice 5 Step 0.5: streamable-HTTP transport over the internal Docker
# bridge. `sift-mcp` is the long-lived FastMCP endpoint; `MCP_TRANSPORT_TOKEN`
# is the shared bearer pinned in .env / docker-compose.yaml.
MCP_URL = "http://sift-mcp:8000/mcp"
MCP_HEADERS = {"Authorization": f"Bearer {os.environ['MCP_TRANSPORT_TOKEN']}"}

raw_results_path = Path("out/raw_results.jsonl")
raw_results_path.unlink(missing_ok=True)  # fresh run per execute

raw_by_step: dict[int, RawResult] = {}
failed_step: int | None = None

with propagate_attributes(
    session_id=pipeline_state.run_id,
    user_id=CASE_ID,
    tags=["phase:execute"],
    metadata={"phase": "execute", "n_steps_planned": len(PLAN.steps)},
):
    with langfuse.start_as_current_observation(name="execute", as_type="span") as exec_span:
        async with streamablehttp_client(MCP_URL, headers=MCP_HEADERS) as (read, write, _get_session_id):
            async with ClientSession(read, write) as session:
                await session.initialize()

                for step in PLAN.steps:
                    try:
                        resolved = _resolve_args(step.args, raw_by_step)
                    except ResolverError as e:
                        print(f"  step {step.step_id:>2}  {step.tool:<14}  resolve FAIL  {e}")
                        exec_span.update(level="ERROR", status_message=f"resolve step {step.step_id}: {e}")
                        failed_step = step.step_id
                        break

                    with langfuse.start_as_current_observation(
                        name=step.tool,
                        as_type="tool",
                        input={"step_id": step.step_id, "args": resolved, "purpose": step.purpose},
                    ) as tool_span:
                        # Slice 5 Step 0.5: MCP server is long-lived and case-agnostic;
                        # inject case scope per call so the audit trail lands under <CASE_ID>/analysis/.
                        resolved["case_id"] = CASE_ID
                        r = _unwrap_mcp(await session.call_tool(step.tool, resolved))
                        raw = RawResult(
                            step_id=step.step_id,
                            tool_call_id=r["tool_call_id"],
                            tool=step.tool,
                            args=resolved,
                            exit_code=r["exit_code"],
                            stdout_excerpt=r["stdout_excerpt"],
                            stdout_path=r["stdout_path"],
                            duration_ms=r["duration_ms"],
                        )
                        raw_by_step[step.step_id] = raw
                        with raw_results_path.open("a", encoding="utf-8") as f:
                            f.write(raw.model_dump_json() + "\n")
                        tool_span.update(
                            output={
                                "exit_code": r["exit_code"],
                                "duration_ms": r["duration_ms"],
                                "stdout_hash": r["stdout_hash"],
                                "stdout_path": r["stdout_path"],
                                "truncated": r["truncated"],
                            },
                            metadata={"tool_call_id": r["tool_call_id"]},
                        )
                        status = "OK  " if r["exit_code"] == 0 else "FAIL"
                        print(
                            f"  step {step.step_id:>2}  {step.tool:<14}  {status}  "
                            f"exit={r['exit_code']}  dur={r['duration_ms']:>5}ms"
                        )
                        if r["exit_code"] != 0:
                            tool_span.update(level="ERROR", status_message=f"exit_code={r['exit_code']}")
                            exec_span.update(level="ERROR", status_message=f"step {step.step_id} failed")
                            failed_step = step.step_id
                            break

        exec_span.update(output={
            "n_steps_executed": len(raw_by_step),
            "n_steps_planned": len(PLAN.steps),
            "failed_step": failed_step,
            "all_exit_zero": all(r.exit_code == 0 for r in raw_by_step.values()),
        })

langfuse.flush()

pipeline_state.raw_results = list(raw_by_step.values())
print()
print(f"executed {len(raw_by_step)}/{len(PLAN.steps)} steps")
print(f"raw_results.jsonl → {raw_results_path.resolve()}")

if failed_step is not None:
    raise RuntimeError(
        f"execute halted at step {failed_step} — see out/raw_results.jsonl for partial output"
    )


Propagated attribute 'metadata.n_steps_planned' value is not a string. Dropping value.


  step  1  fsstat_e01      OK    exit=0  dur=  123ms
  step  2  fls_list        OK    exit=0  dur= 1747ms
  step  3  fls_list        OK    exit=0  dur= 1487ms
  step  4  fls_list        OK    exit=0  dur= 1784ms
  step  5  fls_list        OK    exit=0  dur= 1456ms
  step  6  icat_extract    OK    exit=0  dur=  742ms
  step  7  icat_extract    OK    exit=0  dur=  534ms
  step  8  regripper_run   OK    exit=0  dur=   72ms
  step  9  regripper_run   OK    exit=0  dur=   48ms
  step 10  regripper_run   OK    exit=0  dur=  323ms
  step 11  regripper_run   OK    exit=0  dur=  195ms
  step 12  regripper_run   OK    exit=0  dur=   43ms
  step 13  regripper_run   OK    exit=0  dur=   45ms
  step 14  regripper_run   OK    exit=0  dur=  207ms
  step 15  fls_list        OK    exit=0  dur= 1442ms
  step 16  fls_list        OK    exit=0  dur= 1453ms
  step 17  icat_extract    OK    exit=0  dur=  156ms
  step 18  regripper_run   OK    exit=0  dur=   53ms

executed 18/18 steps
raw_results.jsonl → /wor

## C9 — INTERPRET (real implementation)

Final phase: read the 18 raw tool outputs from `pipeline_state.raw_results`, have claude-sonnet-4.6 synthesize them into a `Findings` report per the Pydantic schema in C2.

**Contract:**
- Model: `anthropic/claude-sonnet-4.6` (via `MODELS["interpret"]`); cache_control on system prompt per the default-caching rule.
- The LLM emits only `{"findings": [...]}`. We own `case_id`, `question`, `plan_digest`, `started_at`, `finished_at` — model can't fabricate them.
- `plan_digest` = SHA-256 of `out/tool_plan.json` bytes. Locks the finding back to the exact plan the human approved.
- Writes `out/findings.json` (pretty-printed `Findings.model_dump_json`).
- **Writes `out/findings.SUCCESS` only on clean end-to-end completion.** That file is the machine-readable "this was a good run" marker — Slice 3 Critic / portfolio tooling filter on its existence. Manually starring sessions in Langfuse is the parallel human signal.
- Fourth trace `interpret` in the same Langfuse session; parent span carries the Pydantic-dumped Findings + metadata (`n_findings`, `n_high_confidence`, `plan_digest`).

**Evidence contract:** every `Finding.evidence[i]` must reference a `tool_call_id` that appears in `raw_results` AND the `output_excerpt` must be a literal quote from that step's stdout. Prompt enforces this. Fabricated evidence would be a Slice-2-blocking bug; we'll sanity-check after the first run.


In [10]:
import hashlib
import json
from datetime import datetime, timezone
from pathlib import Path

from langfuse import propagate_attributes

assert pipeline_state.raw_results, "pipeline_state.raw_results missing — run C8 first."
assert pipeline_state.tool_plan is not None, "pipeline_state.tool_plan missing — run C6 first."

# SHA-256 of the approved plan — locks findings to the exact plan the human OK'd.
plan_bytes = Path("out/tool_plan.json").read_bytes()
plan_digest = hashlib.sha256(plan_bytes).hexdigest()

plan_steps_by_id = {s.step_id: s for s in pipeline_state.tool_plan.steps}

# Assemble the per-step bundle for the LLM. We send stdout_excerpt (MCP server
# already 64KB-capped) rather than full stdout_path contents — regripper output
# is always small and fls bodyfiles are only evidence for navigation, not
# persistence. Cheaper and faster; if a finding needs more context we can
# re-read stdout_path later.
bundle = {
    "question": pipeline_state.tool_plan.question,
    "case_id": CASE_ID,
    "steps": [
        {
            "step_id": r.step_id,
            "tool_call_id": r.tool_call_id,
            "tool": r.tool,
            "purpose": plan_steps_by_id[r.step_id].purpose,
            "args": r.args,
            "exit_code": r.exit_code,
            "stdout_excerpt": r.stdout_excerpt,
        }
        for r in pipeline_state.raw_results
    ],
}

INTERPRET_SYSTEM_PROMPT = """You are a DFIR (digital forensics and incident response) analyst. You receive the outputs of tool calls run against a Windows E01 disk image (`fsstat`, `fls`, `icat`, `regripper`) and the original investigation question. Your job is to produce a Findings JSON.

## Hard rules

1. **Evidence must be real.** Every `Finding.evidence[i]` entry must have:
   - a `tool_call_id` that appears in the bundle's `steps[*].tool_call_id`
   - an `output_excerpt` that is a literal substring of that step's `stdout_excerpt`
   NEVER fabricate registry keys, service names, paths, or values.

2. **Only report what the tools show.** A Finding means you have stdout-backed evidence of a persistence mechanism. Do not emit a Finding because "this key usually exists" or "this plugin typically returns X". No evidence → no Finding.

3. **Classify every finding.** Every `Finding` MUST set the `classification` field to one of:
   - `attacker_persistence`       — confidently malicious; `notes` must explicitly rule out benign alternatives (see Disambiguation below)
   - `legitimate_responder_tool`  — DFIR/IR tool installed during incident response
   - `legitimate_vendor_product`  — commercial security or IT product
   - `legitimate_windows_default` — stock Windows component or driver (also use for `NOT_FOUND` findings)
   - `requires_disambiguation`    — signals suggest malicious but you cannot rule out benign; emit as MEDIUM confidence with unresolved alternatives in `notes`
   Findings classified as `legitimate_*` should NOT be emitted unless the caller explicitly asked for an inventory (they are not findings in the investigative sense).

4. **If nothing suspicious is found, emit exactly one Finding with** category="NOT_FOUND", mechanism="none", value="", evidence=[], confidence="high", classification="legitimate_windows_default".

5. **`confidence`** reflects how strongly the evidence implicates persistence:
   - `high`: clear suspicious path + value + category alignment + benign alternatives ruled out
   - `medium`: plausible but could be legitimate; worth deeper review
   - `low`: weak signal; flagging for completeness

## Disambiguation requirement (read carefully — Slice 2.5 surfaced this as the dominant failure class)

Before classifying any mechanism as `attacker_persistence`, you MUST rule out benign explanations. A mechanism is NOT attacker persistence if it is:

  **(a) A DFIR / incident-response tool installed by responders.** Ask: does the name, path, or command line match a known forensics product? Examples of DFIR tool signatures (non-exhaustive):
    - F-Response      (`subject_srv.exe`; connects to `*-hunt.*` or `*-examiner.*` hosts on high non-standard ports)
    - Mnemosyne       (`Mnemosyne.sys` kernel driver — memory acquisition)
    - Volatility / `vol.py` (memory analysis)
    - KAPE            (`kape.exe`; Targets/Modules structure)
    - Velociraptor    (`velociraptor.exe`; endpoint agent)
    - Magnet AXIOM, MemProcFS, WinPMEM, DumpIt, FTK Imager, Redline, CyLR, Kansa
    - Sysmon / SysmonDrv — legitimate by default, but note that attackers occasionally install Sysmon for their own monitoring; flag the unusual case rather than auto-exonerate.

  **(b) A commercial security or IT product.** McAfee (`mfe*`, `McAfeeFramework`, `McShield`, `enterceptAgent`, `HipMgmt`, `HipShieldK`), CrowdStrike, SentinelOne, Symantec, Trend Micro, VMware guest tools (`VMTools`, `VGAuthService`, `VMMemCtl`, `vmware-*`), VirtualBox guest, Microsoft Defender (`WinDefend`, `MpsSvc`, `WdNisSvc`), Windows Update (`wuauserv`), `AdobeARMservice`, GoogleUpdate (`gupdate` / `gupdatem`), `MozillaMaintenance`.

  **(c) A Windows default or a legitimate vendor driver.** Perf* services (`PerfDisk`, `PerfHost`, `PerfNet`, `PerfOS`, `PerfProc`), RPC family (`RpcEptMapper`, `RpcSs`, `DcomLaunch`), TCP/IP stack (`Tcpip`, `NetBT`, `NetBIOS`), kernel drivers for storage / input / USB / virtual hardware (`atapi`, `usbhub`, `i8042prt`, `vmbus`, `storvsc`, etc.), `.NET`/`ASP.NET` service family, clr_optimization_*, `aspnet_state`.

**Masquerading counter-rule:** if a name mimics a Windows built-in but the binary/path is NOT the standard one (e.g., a service named "PerfMon" running `perfmonsvc64.exe` when the legitimate Windows perf services are `PerfDisk`, `PerfHost`, `PerfNet`, `PerfOS`, `PerfProc`), that is **evidence of masquerading** and overrides the "looks like Windows default" heuristic. Classify as `attacker_persistence` with notes explaining the name/path mismatch.

**For every `attacker_persistence` finding at `high` confidence, `notes` MUST contain the benign hypotheses you considered and ruled out** — even briefly. Example: "Ruled out DFIR tools (not a known responder product), vendor products (not in McAfee/VMware/Defender path conventions), Windows defaults (not among Perf*/RPC/TCP-IP service families). Binary path under C:\\windows\\ with non-Microsoft name and C2-like outbound connection pattern."

## Output

Emit exactly:

```json
{
  "findings": [
    {
      "category": "<PersistenceCategory literal>",
      "mechanism": "<short human label, e.g. 'HKLM Run key', 'Windows service auto-start'>",
      "value": "<the suspicious path/command/value string>",
      "confidence": "low|medium|high",
      "classification": "attacker_persistence|legitimate_responder_tool|legitimate_vendor_product|legitimate_windows_default|requires_disambiguation",
      "evidence": [
        {"tool_call_id": "<from bundle>", "output_excerpt": "<literal quote from stdout>"}
      ],
      "notes": "<for attacker_persistence: which benign hypotheses you ruled out; for requires_disambiguation: what unresolved alternatives remain>"
    }
  ]
}
```

`category` must be one of: `registry_run_key`, `service`, `scheduled_task`, `ifeo_debugger`, `appinit_dll`, `logon_script`, `NOT_FOUND`.

`classification` must be one of the five values listed in Hard Rule 3. DO NOT emit `legitimate_responder_tool`, `legitimate_vendor_product`, or `legitimate_windows_default` findings unless you are compiling an inventory — those are suppressed, not reported. The exception is the single `NOT_FOUND` finding (Hard Rule 4) which uses `classification="legitimate_windows_default"`.
"""

started_at = datetime.now(timezone.utc)

with propagate_attributes(
    session_id=pipeline_state.run_id,
    user_id=CASE_ID,
    tags=["phase:interpret"],
    metadata={"phase": "interpret", "n_steps": len(pipeline_state.raw_results)},
):
    with langfuse.start_as_current_observation(name="interpret", as_type="span") as interpret_span:
        # `extract_client` is the Langfuse-wrapped OpenAI client; cache_control on
        # the system block keeps repeat runs cheap per the default-caching rule.
        # Slice 3 Phase B: see C6 for the rationale — corrective goes in a
        # separate system block so the cached first block stays byte-identical
        # on first runs and cache-hits remain cheap.
        messages = [
            {"role": "system", "content": [
                {"type": "text", "text": INTERPRET_SYSTEM_PROMPT,
                 "cache_control": {"type": "ephemeral"}},
            ]},
        ]
        if pipeline_state.corrective_instruction:
            messages.append({
                "role": "system",
                "content": f"CRITIC CORRECTION (retry pass)\n\n{pipeline_state.corrective_instruction}",
            })
        messages.append({"role": "user", "content": json.dumps(bundle, indent=2)})
        resp = extract_client.chat.completions.create(
            model=MODELS["interpret"],
            messages=messages,
            response_format={"type": "json_object"},
            max_tokens=8000,
        )
        raw = resp.choices[0].message.content

        # `_parse_json_response` is defined in C6 and handles Claude's occasional
        # ```json fences. Requires C6 to have been run earlier in this kernel.
        import re as _re
        _s = raw.strip()
        if _s.startswith("```"):
            _s = _re.sub(r"^```(?:json|JSON)?\s*", "", _s)
            _s = _re.sub(r"\s*```\s*$", "", _s)
        parsed = json.loads(_s)

        # Validate the model's Finding entries individually.
        # Post-Step-0 (2026-04-19): `classification` is now a required field.
        # If the model omits it, Pydantic raises here — that's the intended
        # early-failure signal that the disambiguation prompt isn't landing.
        # Once Slice 3 Critic R_11 ships, this becomes a soft retry instead.
        finding_objs = [Finding.model_validate(f) for f in parsed.get("findings", [])]

        finished_at = datetime.now(timezone.utc)

        # Assemble the full Findings object — we own the metadata, model owns findings.
        findings = Findings(
            case_id=CASE_ID,
            question=pipeline_state.tool_plan.question,
            findings=finding_objs,
            plan_digest=plan_digest,
            started_at=started_at,
            finished_at=finished_at,
        )

        # Sanity-check evidence references real tool_call_ids. Soft-warn only —
        # a strict validator here belongs in the Critic (Slice 3), not here.
        valid_ids = {r.tool_call_id for r in pipeline_state.raw_results}
        dangling = [
            (i, e.tool_call_id)
            for i, f in enumerate(findings.findings)
            for e in f.evidence
            if e.tool_call_id not in valid_ids
        ]
        if dangling:
            print(f"WARN: {len(dangling)} evidence entries reference unknown tool_call_ids — {dangling[:3]}")

        interpret_span.update(
            output=findings.model_dump(mode="json"),
            metadata={
                "n_findings": len(findings.findings),
                "n_high_confidence": sum(1 for f in findings.findings if f.confidence == "high"),
                "n_attacker_persistence": sum(1 for f in findings.findings if f.classification == "attacker_persistence"),
                "n_requires_disambiguation": sum(1 for f in findings.findings if f.classification == "requires_disambiguation"),
                "plan_digest_short": plan_digest[:16],
            },
        )

# Persist — findings.json is the real artifact, findings.SUCCESS is the marker.
Path("out/findings.json").write_text(findings.model_dump_json(indent=2), encoding="utf-8")
Path("out/findings.SUCCESS").touch()

pipeline_state.findings = findings
langfuse.flush()

print(f"findings: {len(findings.findings)}")
for f in findings.findings:
    print(f"  [{f.confidence:>6}] [{f.classification:<28}] {f.category:<20}  {f.mechanism}: {f.value[:70]}")
print()
print(f"plan_digest:         {plan_digest[:16]}…")
print(f"out/findings.json    → {Path('out/findings.json').resolve()}")
print(f"out/findings.SUCCESS → {Path('out/findings.SUCCESS').resolve()}")


Propagated attribute 'metadata.n_steps' value is not a string. Dropping value.


findings: 2
  [  high] [attacker_persistence        ] service               Windows service auto-start (masquerading as PerfMon): c:\windows\system32\perfmonsvc64.exe
  [  high] [attacker_persistence        ] service               Windows service — named-pipe beacon via cmd.exe: %COMSPEC% /c echo b6a1458f396 > \\.\pipe\334485

plan_digest:         52c5f0d799098393…
out/findings.json    → /workspace/out/findings.json
out/findings.SUCCESS → /workspace/out/findings.SUCCESS


## C10 — Critic rules (deterministic)

Stateless Critic subagent per [`slice-3-runbook.md`](../../docs/runbooks/slice-3-runbook.md) Step 2. Eleven pure-Python rules, each answering one plain-English question about a `Finding`.

- **R_01–R_10** check structural integrity — is the finding well-formed, grounded in real evidence, produced by the right tools, exit-code-clean, not fabricated.
- **R_11** checks semantic correctness — did the agent declare what *kind* of thing this finding is (attacker persistence vs. responder tool vs. vendor product vs. Windows default).

The Critic operates on the `Finding` + raw tool-call results + the approved plan. It does **not** see the Interpret agent's chain-of-thought or other Findings — by design. This is the architectural guard against indirect prompt injection: even if INTERPRET was poisoned, the Critic re-grounds against bytes.

**Severity contract:**
- **R_05** (EXCERPT_HALLUCINATION) and **R_10** (INJECTION_FLAGGED_EVIDENCE) always escalate — no retry. Excerpt fabrication and adversarial evidence are integrity failures, not reasoning failures.
- All other rules trigger retry with a per-rule `new_instruction` correction template (wired in C12).

In [ ]:
# Critic rules + orchestrator + retry policy extracted to pipeline/critic.py
# (Slice 5 Step 1). The original C10/C11/C12 bodies live there now.
#
# Imported below:
#   CriticContext, CATEGORY_REQUIRED_TOOLS
#   R_01, R_02, R_03, R_04, R_05, R_06, R_07, R_08, R_09, R_10, R_11, R_12, R_13
#   CRITIC_RULES, ESCALATE_CODES
#   critic_evaluate
#   PER_FINDING_RETRY_LIMIT, TOKEN_CEILING_PER_INVESTIGATION, total_roundtrip_limit
#   NEW_INSTRUCTION_TEMPLATES, RETRY_BRANCH, build_new_instruction, critic_edge
from pipeline.critic import *

print(f"Loaded {len(CRITIC_RULES)} Critic rules: {[r.__name__ for r in CRITIC_RULES]}")
print(f"Escalate-only failure codes: {sorted(ESCALATE_CODES)}")


## C10b — Rule unit tests (fixture-based)

Per runbook Step 2: *'each rule fires on a hand-crafted bad finding and passes on a good one.'* Single-cell harness — each rule gets a (`bad`, `good`) pair; assertions print ✓ or fail loudly.

Tests run in-process with synthetic `RawResult` and `Finding` objects; no MCP, no real E01 required.

In [ ]:
# ---- Test fixtures ----

def _mk_raw(tool_call_id="tc-1", tool="regripper_run", exit_code=0,
            stdout="Run\\updater -> C:\\Users\\public\\updater.exe",
            step_id=1, args=None, duration_ms=10, injection_flagged=False):
    r = RawResult(step_id=step_id, tool_call_id=tool_call_id, tool=tool,
                  args=args or {}, exit_code=exit_code,
                  stdout_excerpt=stdout,
                  stdout_path=f"/tmp/does-not-exist-{tool_call_id}.stdout",
                  duration_ms=duration_ms)
    if injection_flagged:
        # Bypass Pydantic's attribute guard — the real flag ships in Slice 5.
        object.__setattr__(r, "injection_flagged", True)
    return r

def _mk_plan(): return ToolPlan(question="test", steps=[], expected_findings_range=(0, 10))
def _mk_ctx(raws): return CriticContext(tool_plan=_mk_plan(), raw_results=raws)

def _mk_finding(category="registry_run_key", classification="attacker_persistence",
                mechanism=r"HKLM\Software\Microsoft\Windows\CurrentVersion\Run\updater",
                value=r"C:\Users\public\updater.exe", confidence="high",
                evidence=None, notes="ruled out DFIR-responder / vendor / Windows-default alternatives"):
    return Finding(
        category=category, mechanism=mechanism, value=value, confidence=confidence,
        classification=classification,
        evidence=evidence or [Evidence(tool_call_id="tc-1",
                                       output_excerpt=r"Run\updater -> C:\Users\public\updater.exe")],
        notes=notes,
    )

def _check(rule, bad, good, *, name=None):
    name = name or rule.__name__
    bf, bc = bad
    gf, gc = good
    br = rule(bf, bc)
    gr = rule(gf, gc)
    assert br is not None, f"{name}: expected FAIL on bad finding, got PASS"
    assert br.rule_id == name, f"{name}: fired rule_id={br.rule_id}"
    assert gr is None, f"{name}: expected PASS on good finding, got {gr}"
    print(f"  {name} ✓ fires on bad ({br.code}), passes on good")

# Shared good context/finding for most rules
_good_raw_regripper = _mk_raw(tool="regripper_run")
_good_ctx = _mk_ctx([_good_raw_regripper])
_good_finding = _mk_finding()

print("Running R_01–R_12 unit tests (R_13 is a Slice-5 stub — separately asserted):")
print()

# R_01 — cite a tool_call_id that doesn't exist
_check(R_01,
    bad=(_mk_finding(evidence=[Evidence(tool_call_id="tc-NOPE", output_excerpt="x")]), _good_ctx),
    good=(_good_finding, _good_ctx))

# R_02 — mechanism/value tokens absent from all excerpts
_bad_ctx_02 = _mk_ctx([_mk_raw(stdout="totally unrelated bytes here")])
_bad_f_02 = _mk_finding(
    mechanism=r"HKLM\Software\UniqueNonsense\Run\xyznonsense",
    value=r"C:\unlikely\xyznonsense.exe",
    evidence=[Evidence(tool_call_id="tc-1", output_excerpt="totally unrelated bytes here")])
_check(R_02, bad=(_bad_f_02, _bad_ctx_02), good=(_good_finding, _good_ctx))

# R_03 — registry_run_key finding citing ONLY fls_list (wrong tool for the category)
_bad_ctx_03 = _mk_ctx([_mk_raw(tool="fls_list")])
_check(R_03, bad=(_good_finding, _bad_ctx_03), good=(_good_finding, _good_ctx))

# R_04 — registry_run_key mechanism that doesn't start with HKLM/HKCU
_bad_f_04 = _mk_finding(mechanism="some descriptive label, not a registry path")
_check(R_04, bad=(_bad_f_04, _good_ctx), good=(_good_finding, _good_ctx))

# R_05 — excerpt claims bytes not in the actual stdout (fabrication)
_bad_ctx_05 = _mk_ctx([_mk_raw(stdout="real bytes here only")])
_bad_f_05 = _mk_finding(evidence=[Evidence(tool_call_id="tc-1",
                                            output_excerpt="FABRICATED QUOTE NOT IN STDOUT")])
_check(R_05, bad=(_bad_f_05, _bad_ctx_05), good=(_good_finding, _good_ctx))

# R_06 — NOT_FOUND at high confidence but required tools never ran successfully
_bad_ctx_06 = _mk_ctx([_mk_raw(tool="fsstat_e01")])  # regripper never ran
_bad_f_06 = Finding(category="NOT_FOUND", mechanism="none", value="", confidence="high",
                    classification="legitimate_windows_default", evidence=[], notes="")
# Good: the full tool set ran ok
_good_ctx_06 = _mk_ctx([_mk_raw(tool=t, tool_call_id=f"tc-{i}")
                        for i, t in enumerate(["fsstat_e01", "fls_list", "icat_extract", "regripper_run"])])
_good_f_06 = _bad_f_06
_check(R_06, bad=(_bad_f_06, _bad_ctx_06), good=(_good_f_06, _good_ctx_06))

# R_07 — non-NOT_FOUND finding with empty mechanism
_bad_f_07 = _mk_finding(mechanism="")
_check(R_07, bad=(_bad_f_07, _good_ctx), good=(_good_finding, _good_ctx))

# R_08 — high-confidence finding with no primary-tool evidence (only fls_list cited)
_bad_ctx_08 = _mk_ctx([_mk_raw(tool="fls_list")])  # not a primary tool for registry_run_key
# Have to bypass R_03 first — create a finding whose evidence cites fls but we still claim high conf
# R_08 fires independently of R_03; test will pass since we just check R_08 alone
_check(R_08, bad=(_good_finding, _bad_ctx_08), good=(_good_finding, _good_ctx))

# R_09 — evidence cites a tool call with non-zero exit_code
_bad_ctx_09 = _mk_ctx([_mk_raw(exit_code=1)])
_check(R_09, bad=(_good_finding, _bad_ctx_09), good=(_good_finding, _good_ctx))

# R_10 — evidence cites a tool call flagged by the injection scanner
_bad_ctx_10 = _mk_ctx([_mk_raw(injection_flagged=True)])
_check(R_10, bad=(_good_finding, _bad_ctx_10), good=(_good_finding, _good_ctx))

# R_11 — attacker_persistence at high confidence with notes missing ruled-out language
_bad_f_11 = _mk_finding(notes="just a plain note with no alternatives mentioned")
_check(R_11, bad=(_bad_f_11, _good_ctx), good=(_good_finding, _good_ctx))

# R_12 — NOT_FOUND at high confidence but a tool elsewhere in the run failed.
# Additive over R_06: R_06 checks the category-required tool set; R_12 checks
# the whole run. A high-confidence "nothing found" is only defensible if the
# run itself was clean.
_bad_ctx_12 = _mk_ctx([
    _mk_raw(tool="regripper_run", tool_call_id="tc-rr",  exit_code=0),
    _mk_raw(tool="fls_list",      tool_call_id="tc-fls", exit_code=2),  # <- broken
    _mk_raw(tool="icat_extract",  tool_call_id="tc-ica", exit_code=0),
])
_bad_f_12 = Finding(category="NOT_FOUND", mechanism="none", value="", confidence="high",
                    classification="legitimate_windows_default", evidence=[], notes="")
_good_ctx_12 = _mk_ctx([
    _mk_raw(tool="regripper_run", tool_call_id="tc-rr",  exit_code=0),
    _mk_raw(tool="fls_list",      tool_call_id="tc-fls", exit_code=0),
    _mk_raw(tool="icat_extract",  tool_call_id="tc-ica", exit_code=0),
])
_good_f_12 = _bad_f_12
_check(R_12, bad=(_bad_f_12, _bad_ctx_12), good=(_good_f_12, _good_ctx_12))

# R_13 — STUB (pre-Slice-5). _check() expects fire-on-bad, which the stub
# intentionally never does. Assert the no-op contract directly.
assert R_13(_good_finding, _good_ctx) is None, "R_13 stub must return None on good"
assert R_13(_bad_f_05, _bad_ctx_05) is None, "R_13 stub must return None on R_05-style fabrication too"
print("  R_13 ✓ stub no-op — real check deferred to Slice 5 (structured hive_lastwrite)")

print()
print("All 13 rules registered (R_01–R_12 active, R_13 stub pending Slice 5) ✓")

## C11 — Critic orchestrator (`critic_evaluate`)

Runs the 11 rules in order on a single `Finding`, aggregates failures into a `CritiqueResult`, and decides severity:

- **pass**  — all rules returned `None`
- **escalate** — any failure whose code is in `ESCALATE_CODES` (R_05 fabrication, R_10 adversarial evidence)
- **retry** — any other failure

**Deterministic only for v1** per runbook. LLM fallback layer deferred to Slice 3.5 (only if 2.5+ evals demand it — they don't, post-Step-0).

In [ ]:
# `critic_evaluate` now lives in pipeline.critic (imported via C10).
# This cell keeps the live-kernel smoke test that can run standalone
# if `pipeline_state` has a completed run.

if "pipeline_state" in globals() and getattr(pipeline_state, "findings", None) is not None \
        and pipeline_state.raw_results and pipeline_state.tool_plan is not None:
    _ctx = CriticContext(pipeline_state.tool_plan, pipeline_state.raw_results)
    _results = [critic_evaluate(f, _ctx, i) for i, f in enumerate(pipeline_state.findings.findings)]
    print(f"Critic ran on {len(_results)} findings from pipeline_state:")
    for r in _results:
        badges = ", ".join(f"{rf.rule_id}={rf.code}" for rf in r.rules_failed) or "–"
        print(f"  finding[{r.finding_index}] severity={r.severity:<8}  passed={len(r.rules_passed)}/13  failed=[{badges}]")
else:
    print("(no pipeline_state available — run C1 → C9 first, then re-run this cell)")


## C12 — Retry policy + `new_instruction` templates (Step 4 / 4a)

Three pieces:

1. **Retry-budget constants** — `PER_FINDING_RETRY_LIMIT = 2`, `TOKEN_CEILING_PER_INVESTIGATION = 200_000`, `total_roundtrip_limit = min(2 * len(plan.steps), 15)`.
2. **Per-rule `new_instruction` templates** — each `FailureCode` maps to a callable that renders a targeted correction message for the upstream re-dispatch. R_05 and R_10 have no templates (they escalate, never retry).
3. **`critic_edge`** — the LangGraph branch function. Given `PipelineState`, returns one of `commit` / `re_interpret` / `re_plan` / `escalate`.

**Integration note:** this cell defines the machinery. Wiring the nodes into [C4](#)'s `StateGraph.add_conditional_edges` is a separate focused surgery (notes at the bottom of this runbook pass). For now, components are import-ready — scenarios in C14 exercise them without needing the full graph.

In [ ]:
# Retry policy + new_instruction templates + critic_edge now live in
# pipeline.critic (imported via C10). This cell prints the current policy
# summary as a live-kernel sanity check — the logic itself is in the module.

print(f"Retry policy: per-finding={PER_FINDING_RETRY_LIMIT}, token ceiling={TOKEN_CEILING_PER_INVESTIGATION:,}")
print(f"new_instruction templates: {sorted(NEW_INSTRUCTION_TEMPLATES)} ({len(NEW_INSTRUCTION_TEMPLATES)} codes)")
print(f"escalate-only codes (no template): {sorted(ESCALATE_CODES)}")


## C13 — Audit-trail writer (`critic_disagreements.jsonl`)

Appends one line per disagreement to the per-case audit log. **Only called on retry or escalate paths** — a passing Critic run leaves the file untouched (so empty file = no disagreements this case).

Artifact shape matches [`slice-3-runbook.md`](../../docs/runbooks/slice-3-runbook.md#step-5--audit-trail-writer-c13) Step 5. Slice 6 adds sha256 chain-of-custody attestation on top of this file.

In [ ]:
import json as _json
from datetime import datetime, timezone
from pathlib import Path


def append_critic_disagreement(
    disagreements_path: Path,
    *, plan_digest: str, iteration: int, original_finding: Finding,
    critique: CritiqueResult, resolution: dict, cost_so_far: dict,
) -> None:
    """Append one JSONL line per Critic disagreement. No-op via caller for pass.
    resolution shape: {action: retry|escalate, strategy: re_interpret|re_plan|human_review,
                       new_instruction: str|None}
    cost_so_far shape: {input_tokens: int, output_tokens: int, usd_estimate: float|None}"""
    event = CriticDisagreement(
        plan_digest=plan_digest,
        iteration=iteration,
        original_finding=original_finding,
        critic_critique=critique,
        resolution=resolution,
        cost_so_far=cost_so_far,
        timestamp_utc=datetime.now(timezone.utc),
    )
    disagreements_path.parent.mkdir(parents=True, exist_ok=True)
    with disagreements_path.open("a", encoding="utf-8") as fh:
        fh.write(event.model_dump_json() + "\n")


def build_resolution(critique: CritiqueResult, finding: Finding,
                     ctx: CriticContext) -> dict:
    """Build the resolution payload for the audit entry based on the critique's severity
    and which specific rules failed. Resolves action + strategy + new_instruction together."""
    if critique.severity == "pass":
        return {"action": "commit", "strategy": None, "new_instruction": None}
    if critique.severity == "escalate":
        return {"action": "escalate", "strategy": "human_review", "new_instruction": None}
    # retry — combine corrections across all retryable failures
    retryable = [rf for rf in critique.rules_failed if rf.code not in ESCALATE_CODES]
    if not retryable:
        return {"action": "escalate", "strategy": "human_review", "new_instruction": None}
    instructions = [build_new_instruction(rf, finding, ctx, critique.finding_index) for rf in retryable]
    # Branch preference: if any rule needs re_plan, go re_plan; else re_interpret
    strategy = "re_plan" if any(RETRY_BRANCH.get(rf.code) == "re_plan" for rf in retryable) \
               else "re_interpret"
    return {"action": "retry", "strategy": strategy, "new_instruction": "\n\n".join(instructions)}


print("Audit writer loaded: append_critic_disagreement + build_resolution")

## C14 — End-to-end scenarios (component-level)

Four scenarios per [`slice-3-runbook.md`](../../docs/runbooks/slice-3-runbook.md#step-6--end-to-end-smoke-c14) Step 6. Runs at component level — full LangGraph retry-loop integration is a separate C4 surgery (flagged at cell end). Each scenario here verifies:

1. **Happy path** — real post-Step-0 findings pass all 11 rules, Critic returns severity=pass, no audit entry.
2. **Forced R_03 disagreement** — corrupted finding cites wrong-category tool → R_03 fires → resolution = re_plan.
3. **Hallucination escalation** — fabricated `output_excerpt` → R_05 fires → severity=escalate, no retry.
4. **Classification self-correction** — Step-0-reverted finding (missing classification rationale) → R_11 fires → resolution = re_interpret with disambiguation instruction.

Audit entries are written to a tmp path so the test is self-contained.

In [ ]:
import tempfile
from pathlib import Path


def _load_case(case_dir: Path):
    findings_obj = Findings.model_validate_json((case_dir / "findings.json").read_text(encoding="utf-8"))
    raws = [RawResult.model_validate_json(line) for line
            in (case_dir / "raw_results.jsonl").read_text(encoding="utf-8").splitlines() if line.strip()]
    plan = ToolPlan.model_validate_json((case_dir / "tool_plan.json").read_text(encoding="utf-8"))
    return findings_obj, raws, plan


# ---- Set up a tmp audit path for this smoke run ----
_tmp_audit = Path(tempfile.gettempdir()) / "critic_disagreements_smoke.jsonl"
_tmp_audit.unlink(missing_ok=True)


# ---- Scenario 1: happy path on post-Step-0 base-wkstn-05 ----
print("=== Scenario 1: happy path (base-wkstn-05, post-Step-0) ===")
s1_case = Path("out/runs/srl-2018-wkstn-05")
s1_findings, s1_raws, s1_plan = _load_case(s1_case)
s1_ctx = CriticContext(s1_plan, s1_raws)
s1_results = [critic_evaluate(f, s1_ctx, i) for i, f in enumerate(s1_findings.findings)]
assert all(c.severity == "pass" for c in s1_results), f"expected all pass; got {[c.severity for c in s1_results]}"
for i, c in enumerate(s1_results):
    print(f"  finding[{i}] severity={c.severity} passed={len(c.rules_passed)}/11")
print("  ✓ Scenario 1: all findings pass all rules")
print()


# ---- Scenario 2: forced R_03 TOOL_MISMATCH disagreement ----
print("=== Scenario 2: forced R_03 disagreement ===")
# Take finding 0's tool_call_id and pretend it came from fls_list (wrong tool for registry_run_key)
s2_finding = s1_findings.findings[0].model_copy(update={"category": "registry_run_key"})
# Rewrite the evidence to cite a tool_call_id whose RawResult is fls_list, not regripper
s2_raws = [r.model_copy(update={"tool": "fls_list"}) for r in s1_raws]
s2_ctx = CriticContext(s1_plan, s2_raws)
s2_result = critic_evaluate(s2_finding, s2_ctx, 0)
assert s2_result.severity == "retry", f"expected retry; got {s2_result.severity}"
assert any(rf.code == "TOOL_MISMATCH" for rf in s2_result.rules_failed), \
    f"expected TOOL_MISMATCH; got {[rf.code for rf in s2_result.rules_failed]}"
s2_resolution = build_resolution(s2_result, s2_finding, s2_ctx)
assert s2_resolution["strategy"] == "re_plan", f"expected re_plan; got {s2_resolution['strategy']}"
assert "regripper_run" in (s2_resolution["new_instruction"] or ""), \
    "new_instruction should name the missing primary tool"
append_critic_disagreement(_tmp_audit, plan_digest="sha256:demo", iteration=0,
                           original_finding=s2_finding, critique=s2_result,
                           resolution=s2_resolution,
                           cost_so_far={"input_tokens": 1000, "output_tokens": 200, "usd_estimate": None})
print(f"  severity=retry, strategy=re_plan, audit entry written")
print(f"  new_instruction preview: {s2_resolution['new_instruction'][:120]}...")
print("  ✓ Scenario 2: R_03 fires, resolution=re_plan")
print()


# ---- Scenario 3: hallucination escalation ----
print("=== Scenario 3: hallucination escalation ===")
# Fabricate an output_excerpt that doesn't appear in any cited stdout
s3_finding = s1_findings.findings[0].model_copy(update={
    "evidence": [Evidence(tool_call_id=s1_findings.findings[0].evidence[0].tool_call_id,
                          output_excerpt="FABRICATED QUOTE NOT IN STDOUT 12345 xyzzy")]
})
s3_result = critic_evaluate(s3_finding, s1_ctx, 0)
assert s3_result.severity == "escalate", f"expected escalate; got {s3_result.severity}"
assert any(rf.code == "EXCERPT_HALLUCINATION" for rf in s3_result.rules_failed), \
    f"expected EXCERPT_HALLUCINATION; got {[rf.code for rf in s3_result.rules_failed]}"
s3_resolution = build_resolution(s3_result, s3_finding, s1_ctx)
assert s3_resolution["action"] == "escalate", f"expected escalate action; got {s3_resolution}"
assert s3_resolution["new_instruction"] is None, "escalate path must not produce a new_instruction"
append_critic_disagreement(_tmp_audit, plan_digest="sha256:demo", iteration=0,
                           original_finding=s3_finding, critique=s3_result,
                           resolution=s3_resolution,
                           cost_so_far={"input_tokens": 1000, "output_tokens": 200, "usd_estimate": None})
print(f"  severity=escalate, strategy=human_review, no retry")
print("  ✓ Scenario 3: R_05 fires, immediate escalation")
print()


# ---- Scenario 4: classification self-correction (R_11) ----
print("=== Scenario 4: R_11 classification self-correction ===")
# Simulate pre-Step-0 behavior: attacker_persistence at high confidence, notes missing ruled-out language
s4_finding = s1_findings.findings[0].model_copy(update={
    "notes": "just a plain note with no mention of alternatives here"
})
s4_result = critic_evaluate(s4_finding, s1_ctx, 0)
assert s4_result.severity == "retry", f"expected retry; got {s4_result.severity}"
assert any(rf.code == "CLASSIFICATION_MISSING" for rf in s4_result.rules_failed), \
    f"expected CLASSIFICATION_MISSING; got {[rf.code for rf in s4_result.rules_failed]}"
s4_resolution = build_resolution(s4_result, s4_finding, s1_ctx)
assert s4_resolution["strategy"] == "re_interpret", f"expected re_interpret; got {s4_resolution['strategy']}"
assert "disambiguation" in s4_resolution["new_instruction"].lower(), \
    "new_instruction must reference the disambiguation rules"
append_critic_disagreement(_tmp_audit, plan_digest="sha256:demo", iteration=0,
                           original_finding=s4_finding, critique=s4_result,
                           resolution=s4_resolution,
                           cost_so_far={"input_tokens": 1000, "output_tokens": 200, "usd_estimate": None})
print(f"  severity=retry, strategy=re_interpret, disambiguation instruction issued")
print(f"  new_instruction preview: {s4_resolution['new_instruction'][:120]}...")
print("  ✓ Scenario 4: R_11 fires, resolution=re_interpret with disambiguation")
print()


# ---- Audit log final state ----
print("=== Audit log summary ===")
n_entries = sum(1 for _ in _tmp_audit.open("r", encoding="utf-8"))
print(f"  {_tmp_audit} has {n_entries} disagreement entries (expected: 3 — scenarios 2, 3, 4)")
assert n_entries == 3, f"expected 3 entries; got {n_entries}"
print("  ✓ Audit writer wrote one entry per disagreement (scenario 1 happy path: no entry)")
print()
print("ALL 4 SCENARIOS PASS ✓")
print()
print("---")
print("Integration TODO (C4 surgery — deferred):")
print("  - Add 'critic' and 'human_review' nodes to the StateGraph in C4")
print("  - Wire add_conditional_edges('critic', critic_edge, {commit: END, re_interpret: 'interpret',")
print("    re_plan: 'plan', escalate: 'human_review'})")
print("  - Extend PipelineState with iteration, attempts_per_finding, tokens_used, critique_results")
print("  - Thread new_instruction from Critic state into the INTERPRET/PLAN prompt templates on retry")
print("  - This cell's component-level tests prove the pieces work; C4 surgery is the last mile.")